In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 01 — SOURCE AND EXPERIMENT INTEGRITY
# ============================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json

drive.mount("/content/drive")

print("=" * 72)
print("CELL 01 — SOURCE AND EXPERIMENT INTEGRITY")
print("=" * 72)

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")

CIC_E_DIR = PROJECT_ROOT / "results/RF_E_CIC"
CIC_STATS_DIR = PROJECT_ROOT / "results/RF_STATS_CIC"
NSL09_DIR = PROJECT_ROOT / "results/NSL_KDD_PREPROCESSING"
NSL10_DIR = PROJECT_ROOT / "results/NSL_KDD_RF_REPLICATION"

NSL11_RESULTS_DIR = (
    PROJECT_ROOT / "results/CROSS_BENCHMARK_ROBUSTNESS"
)

NSL11_SOURCE_FILES = {
    "Notebook 07 completion":
        CIC_E_DIR / "RF_E_CIC_NOTEBOOK_07_COMPLETE.json",
    "Notebook 07 model alias":
        CIC_E_DIR / "RF_E_CIC_model_alias.json",
    "Notebook 08 completion":
        CIC_STATS_DIR / "RF08_NOTEBOOK_COMPLETE.json",
    "Notebook 08 metrics":
        CIC_STATS_DIR / "RF08_verified_test_metrics.csv",
    "Notebook 08 per-class comparison":
        CIC_STATS_DIR / "RF08_per_class_comparison.csv",
    "Notebook 08 statistical summary":
        CIC_STATS_DIR / "RF08_statistical_summary.json",
    "Notebook 09 completion":
        NSL09_DIR / "NSL09_NOTEBOOK_COMPLETE.json",
    "Notebook 09 manifest":
        NSL09_DIR / "NSL09_manifest.json",
    "Notebook 10 completion":
        NSL10_DIR / "NSL10_NOTEBOOK_COMPLETE.json",
    "Notebook 10 manifest":
        NSL10_DIR / "NSL10_manifest.json",
    "Notebook 10 validation comparison":
        NSL10_DIR / "NSL10_validation_model_comparison.csv",
    "Notebook 10 test comparison":
        NSL10_DIR / "NSL10_official_test_model_comparison.csv",
    "Notebook 10 class comparison":
        NSL10_DIR / "NSL10_official_test_class_comparison.csv",
    "Notebook 10 prediction agreement":
        NSL10_DIR / "NSL10_prediction_agreement.json"
}

missing_files = [
    name for name, path in NSL11_SOURCE_FILES.items()
    if not path.is_file()
]

if missing_files:
    raise RuntimeError(
        "Missing source artefacts: "
        + ", ".join(missing_files)
    )

def ns11_load_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )

NSL11_CIC_COMPLETION = ns11_load_json(
    CIC_STATS_DIR / "RF08_NOTEBOOK_COMPLETE.json"
)

NSL11_NSL09_COMPLETION = ns11_load_json(
    NSL09_DIR / "NSL09_NOTEBOOK_COMPLETE.json"
)

NSL11_NSL10_COMPLETION = ns11_load_json(
    NSL10_DIR / "NSL10_NOTEBOOK_COMPLETE.json"
)

NSL11_NSL09_MANIFEST = ns11_load_json(
    NSL09_DIR / "NSL09_manifest.json"
)

NSL11_NSL10_MANIFEST = ns11_load_json(
    NSL10_DIR / "NSL10_manifest.json"
)

for name, record in [
    ("Notebook 08", NSL11_CIC_COMPLETION),
    ("Notebook 09", NSL11_NSL09_COMPLETION),
    ("Notebook 10", NSL11_NSL10_COMPLETION)
]:
    if record.get("status") != "COMPLETE":
        raise RuntimeError(
            f"{name}: completion status invalid."
        )

expected_nsl_fingerprint = (
    "f321a798f07f753bf0de9455e423175590fd1b000259d3d698a83132aa4f5364"
)

if NSL11_NSL09_MANIFEST[
    "combined_artefact_fingerprint"
] != expected_nsl_fingerprint:
    raise RuntimeError(
        "NSL preprocessing fingerprint mismatch."
    )

if NSL11_NSL10_MANIFEST[
    "source_preprocessing_fingerprint"
] != expected_nsl_fingerprint:
    raise RuntimeError(
        "NSL experiment source fingerprint mismatch."
    )

def ns11_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

# Recheck frozen Notebook 08 result files.

cic_hashes = NSL11_CIC_COMPLETION[
    "output_file_hashes"
]

if len(cic_hashes) != 17:
    raise RuntimeError(
        "Expected 17 CIC statistical artefacts."
    )

for filename, expected_hash in cic_hashes.items():

    path = CIC_STATS_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns11_sha256(path) != expected_hash:
        raise RuntimeError(
            f"CIC result fingerprint mismatch: {filename}"
        )

# Recheck frozen Notebook 10 result files.

nsl_hashes = NSL11_NSL10_MANIFEST[
    "result_file_fingerprints"
]

if len(nsl_hashes) != 22:
    raise RuntimeError(
        "Expected 22 NSL experimental result files."
    )

for relative_path, expected_hash in nsl_hashes.items():

    path = PROJECT_ROOT / relative_path

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns11_sha256(path) != expected_hash:
        raise RuntimeError(
            f"NSL result fingerprint mismatch: {relative_path}"
        )

# Verify frozen model locations.

NSL11_MODEL_PATHS = {
    "RF-B-CIC":
        PROJECT_ROOT / "models/RF_B_CIC/saved_model",
    "RF-E-CIC":
        PROJECT_ROOT / "models/RF_I1_CIC/RF_I1_S/saved_model",
    "RF-B-NSL":
        PROJECT_ROOT / "models/RF_B_NSL/saved_model",
    "RF-H2-NSL":
        PROJECT_ROOT / "models/RF_H2_NSL/saved_model"
}

for name, path in NSL11_MODEL_PATHS.items():

    if not (path / "saved_model.pb").is_file():
        raise RuntimeError(
            f"{name}: SavedModel unavailable at {path}"
        )

print("\nNOTEBOOK COMPLETION RECORDS")
print("Notebook 07: FILE VERIFIED")
print("Notebook 08: COMPLETE")
print("Notebook 09: COMPLETE")
print("Notebook 10: COMPLETE")

print("\nARCHIVED RESULT VERIFICATION")
print("CIC statistical files:", len(cic_hashes))
print("NSL experimental files:", len(nsl_hashes))
print("CIC SHA-256 checks: PASS")
print("NSL SHA-256 checks: PASS")
print("NSL preprocessing fingerprint: PASS")

print("\nMODEL AVAILABILITY")
for name in NSL11_MODEL_PATHS:
    print(f"{name}: VERIFIED")

NSL11_INPUTS_VERIFIED = True

print("\nCELL 01: PASSED")
print("Existing experiments modified: NO")
print("Model training performed: NO")
print("=" * 72)


Mounted at /content/drive
CELL 01 — SOURCE AND EXPERIMENT INTEGRITY

NOTEBOOK COMPLETION RECORDS
Notebook 07: FILE VERIFIED
Notebook 08: COMPLETE
Notebook 09: COMPLETE
Notebook 10: COMPLETE

ARCHIVED RESULT VERIFICATION
CIC statistical files: 17
NSL experimental files: 22
CIC SHA-256 checks: PASS
NSL SHA-256 checks: PASS
NSL preprocessing fingerprint: PASS

MODEL AVAILABILITY
RF-B-CIC: VERIFIED
RF-E-CIC: VERIFIED
RF-B-NSL: VERIFIED
RF-H2-NSL: VERIFIED

CELL 01: PASSED
Existing experiments modified: NO
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON")
print("=" * 72)

if not globals().get("NSL11_INPUTS_VERIFIED", False):
    raise RuntimeError("Cell 01 must pass first.")

NSL11_METRICS_VERIFIED = False

metric_names = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

# Load verified CIC test metrics.

cic_metrics = pd.read_csv(
    CIC_STATS_DIR / "RF08_verified_test_metrics.csv"
).set_index("Model")

if set(cic_metrics.index) != {
    "RF-B-CIC", "RF-E-CIC"
}:
    raise RuntimeError(
        "Unexpected CIC model identifiers."
    )

# Load the two NSL test metric records.

nsl_metric_paths = {
    "RF-B-NSL":
        NSL10_DIR
        / "RF_B_NSL/RF_B_NSL_test_metrics.csv",
    "RF-H2-NSL":
        NSL10_DIR
        / "RF_H2_NSL/RF_H2_NSL_test_metrics.csv"
}

nsl_records = {}

for model, path in nsl_metric_paths.items():

    data = pd.read_csv(path)

    if len(data) != 1:
        raise RuntimeError(
            f"{model}: expected one metric record."
        )

    record = data.iloc[0]

    if record["Model"] != model:
        raise RuntimeError(
            f"{model}: model identifier mismatch."
        )

    if record["Partition"] != "official_test":
        raise RuntimeError(
            f"{model}: partition mismatch."
        )

    if int(record["Records"]) != 22544:
        raise RuntimeError(
            f"{model}: test sample count mismatch."
        )

    nsl_records[model] = record

# Verify archived NSL comparison.

archived_nsl = pd.read_csv(
    NSL10_DIR
    / "NSL10_official_test_model_comparison.csv"
).set_index("Metric")

if set(archived_nsl.index) != set(metric_names):
    raise RuntimeError(
        "NSL test comparison metrics differ."
    )

for metric in metric_names:

    baseline = float(
        nsl_records["RF-B-NSL"][metric]
    )

    tuned = float(
        nsl_records["RF-H2-NSL"][metric]
    )

    for column, expected in [
        ("RF-B-NSL", baseline),
        ("RF-H2-NSL", tuned),
        ("Change", tuned - baseline)
    ]:
        actual = float(
            archived_nsl.loc[metric, column]
        )

        if not np.isclose(
            actual, expected,
            atol=3e-9,
            rtol=0
        ):
            raise RuntimeError(
                f"NSL archived comparison mismatch: "
                f"{metric}, {column}"
            )

# Assemble comparable within-benchmark effects.

experiments = {
    "CIC-IDS2017": {
        "baseline": cic_metrics.loc["RF-B-CIC"],
        "comparison": cic_metrics.loc["RF-E-CIC"],
        "baseline_name": "RF-B-CIC",
        "comparison_name": "RF-E-CIC",
        "intervention": "Depth restriction and training-only SMOTE",
        "classes": 15,
        "test_records": 378119
    },
    "NSL-KDD": {
        "baseline": nsl_records["RF-B-NSL"],
        "comparison": nsl_records["RF-H2-NSL"],
        "baseline_name": "RF-B-NSL",
        "comparison_name": "RF-H2-NSL",
        "intervention": "Frozen maximum-depth restriction only",
        "classes": 5,
        "test_records": 22544
    }
}

model_rows = []
effect_rows = []

for benchmark, experiment in experiments.items():

    for role, name_key in [
        ("Baseline", "baseline_name"),
        ("Comparison", "comparison_name")
    ]:

        source = experiment[
            "baseline" if role == "Baseline"
            else "comparison"
        ]

        row = {
            "Benchmark": benchmark,
            "Model": experiment[name_key],
            "Role": role,
            "Classes": experiment["classes"],
            "Test Records": experiment["test_records"]
        }

        for metric in metric_names:
            row[metric] = float(source[metric])

        model_rows.append(row)

    for metric in metric_names:

        baseline_value = float(
            experiment["baseline"][metric]
        )

        comparison_value = float(
            experiment["comparison"][metric]
        )

        effect_rows.append({
            "Benchmark": benchmark,
            "Baseline": experiment["baseline_name"],
            "Comparison": experiment["comparison_name"],
            "Intervention": experiment["intervention"],
            "Metric": metric,
            "Baseline Value": baseline_value,
            "Comparison Value": comparison_value,
            "Within-Benchmark Change": (
                comparison_value - baseline_value
            )
        })

NSL11_MODEL_METRICS = pd.DataFrame(model_rows)
NSL11_WITHIN_BENCHMARK_EFFECTS = pd.DataFrame(effect_rows)

# Verify known frozen results.

def effect_value(benchmark, metric):
    rows = NSL11_WITHIN_BENCHMARK_EFFECTS.loc[
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Benchmark"]
            == benchmark
        )
        &
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Metric"]
            == metric
        )
    ]

    if len(rows) != 1:
        raise RuntimeError(
            "Expected one effect estimate."
        )

    return float(
        rows.iloc[0]["Within-Benchmark Change"]
    )

if not np.isclose(
    effect_value("CIC-IDS2017", "Macro F1"),
    0.018702,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro F1 effect verification failed."
    )

for metric in metric_names:
    if not np.isclose(
        effect_value("NSL-KDD", metric),
        0.0,
        atol=2e-9,
        rtol=0
    ):
        raise RuntimeError(
            f"Unexpected NSL effect: {metric}"
        )

# Verify paired correctness evidence.

cic_statistical_summary = ns11_load_json(
    CIC_STATS_DIR / "RF08_statistical_summary.json"
)

nsl_agreement = ns11_load_json(
    NSL10_DIR / "NSL10_prediction_agreement.json"
)

mcnemar_p = float(
    cic_statistical_summary["mcnemar"]["p_value"]
)

if not np.isclose(
    mcnemar_p,
    0.5384046,
    atol=1e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC McNemar result mismatch."
    )

if (
    nsl_agreement["validation_prediction_disagreements"]
    != 0
    or nsl_agreement[
        "official_test_prediction_disagreements"
    ] != 0
):
    raise RuntimeError(
        "NSL prediction agreement differs from Notebook 10."
    )

print("\nVERIFIED MODEL PERFORMANCE")

print(
    NSL11_MODEL_METRICS[
        [
            "Benchmark", "Model", "Classes",
            "Test Records", "Accuracy",
            "Macro Recall", "Macro F1", "MCC"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nWITHIN-BENCHMARK EFFECTS")

for benchmark in experiments:

    subset = NSL11_WITHIN_BENCHMARK_EFFECTS.loc[
        NSL11_WITHIN_BENCHMARK_EFFECTS["Benchmark"]
        == benchmark
    ]

    print("\n" + benchmark)

    print(
        subset[
            [
                "Metric",
                "Baseline Value",
                "Comparison Value",
                "Within-Benchmark Change"
            ]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:+.6f}"
        )
    )

print("\nPAIRED PREDICTION VERIFICATION")
print("CIC McNemar p-value:", f"{mcnemar_p:.7f}")
print(
    "NSL validation disagreements:",
    nsl_agreement["validation_prediction_disagreements"]
)
print(
    "NSL test disagreements:",
    nsl_agreement[
        "official_test_prediction_disagreements"
    ]
)

NSL11_METRICS_VERIFIED = True

print("\nCELL 02: PASSED")
print("Within-benchmark changes: VERIFIED")
print("Cross-benchmark raw score equivalence assumed: NO")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON

VERIFIED MODEL PERFORMANCE
  Benchmark     Model  Classes  Test Records  Accuracy  Macro Recall  Macro F1      MCC
CIC-IDS2017  RF-B-CIC       15        378119  0.998606      0.879297  0.893851 0.995364
CIC-IDS2017  RF-E-CIC       15        378119  0.998633      0.927545  0.912553 0.995457
    NSL-KDD  RF-B-NSL        5         22544  0.758384      0.493121  0.504649 0.650888
    NSL-KDD RF-H2-NSL        5         22544  0.758384      0.493121  0.504649 0.650888

WITHIN-BENCHMARK EFFECTS

CIC-IDS2017
           Metric  Baseline Value  Comparison Value  Within-Benchmark Change
         Accuracy       +0.998606         +0.998633                +0.000026
  Macro Precision       +0.935485         +0.901210                -0.034275
     Macro Recall       +0.879297         +0.927545                +0.048248
         Macro F1       +0.893851         +0.912553                +0.018702
      Weighted F1       +0.998570         +0.998638       

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON")
print("=" * 72)

if not globals().get("NSL11_INPUTS_VERIFIED", False):
    raise RuntimeError("Cell 01 must pass first.")

NSL11_METRICS_VERIFIED = False

metric_names = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

# Load verified CIC test metrics.

cic_metrics = pd.read_csv(
    CIC_STATS_DIR / "RF08_verified_test_metrics.csv"
).set_index("Model")

if set(cic_metrics.index) != {
    "RF-B-CIC", "RF-E-CIC"
}:
    raise RuntimeError(
        "Unexpected CIC model identifiers."
    )

# Load the two NSL test metric records.

nsl_metric_paths = {
    "RF-B-NSL":
        NSL10_DIR
        / "RF_B_NSL/RF_B_NSL_test_metrics.csv",
    "RF-H2-NSL":
        NSL10_DIR
        / "RF_H2_NSL/RF_H2_NSL_test_metrics.csv"
}

nsl_records = {}

for model, path in nsl_metric_paths.items():

    data = pd.read_csv(path)

    if len(data) != 1:
        raise RuntimeError(
            f"{model}: expected one metric record."
        )

    record = data.iloc[0]

    if record["Model"] != model:
        raise RuntimeError(
            f"{model}: model identifier mismatch."
        )

    if record["Partition"] != "official_test":
        raise RuntimeError(
            f"{model}: partition mismatch."
        )

    if int(record["Records"]) != 22544:
        raise RuntimeError(
            f"{model}: test sample count mismatch."
        )

    nsl_records[model] = record

# Verify archived NSL comparison.

archived_nsl = pd.read_csv(
    NSL10_DIR
    / "NSL10_official_test_model_comparison.csv"
).set_index("Metric")

if set(archived_nsl.index) != set(metric_names):
    raise RuntimeError(
        "NSL test comparison metrics differ."
    )

for metric in metric_names:

    baseline = float(
        nsl_records["RF-B-NSL"][metric]
    )

    tuned = float(
        nsl_records["RF-H2-NSL"][metric]
    )

    for column, expected in [
        ("RF-B-NSL", baseline),
        ("RF-H2-NSL", tuned),
        ("Change", tuned - baseline)
    ]:
        actual = float(
            archived_nsl.loc[metric, column]
        )

        if not np.isclose(
            actual, expected,
            atol=3e-9,
            rtol=0
        ):
            raise RuntimeError(
                f"NSL archived comparison mismatch: "
                f"{metric}, {column}"
            )

# Assemble comparable within-benchmark effects.

experiments = {
    "CIC-IDS2017": {
        "baseline": cic_metrics.loc["RF-B-CIC"],
        "comparison": cic_metrics.loc["RF-E-CIC"],
        "baseline_name": "RF-B-CIC",
        "comparison_name": "RF-E-CIC",
        "intervention": "Depth restriction and training-only SMOTE",
        "classes": 15,
        "test_records": 378119
    },
    "NSL-KDD": {
        "baseline": nsl_records["RF-B-NSL"],
        "comparison": nsl_records["RF-H2-NSL"],
        "baseline_name": "RF-B-NSL",
        "comparison_name": "RF-H2-NSL",
        "intervention": "Frozen maximum-depth restriction only",
        "classes": 5,
        "test_records": 22544
    }
}

model_rows = []
effect_rows = []

for benchmark, experiment in experiments.items():

    for role, name_key in [
        ("Baseline", "baseline_name"),
        ("Comparison", "comparison_name")
    ]:

        source = experiment[
            "baseline" if role == "Baseline"
            else "comparison"
        ]

        row = {
            "Benchmark": benchmark,
            "Model": experiment[name_key],
            "Role": role,
            "Classes": experiment["classes"],
            "Test Records": experiment["test_records"]
        }

        for metric in metric_names:
            row[metric] = float(source[metric])

        model_rows.append(row)

    for metric in metric_names:

        baseline_value = float(
            experiment["baseline"][metric]
        )

        comparison_value = float(
            experiment["comparison"][metric]
        )

        effect_rows.append({
            "Benchmark": benchmark,
            "Baseline": experiment["baseline_name"],
            "Comparison": experiment["comparison_name"],
            "Intervention": experiment["intervention"],
            "Metric": metric,
            "Baseline Value": baseline_value,
            "Comparison Value": comparison_value,
            "Within-Benchmark Change": (
                comparison_value - baseline_value
            )
        })

NSL11_MODEL_METRICS = pd.DataFrame(model_rows)
NSL11_WITHIN_BENCHMARK_EFFECTS = pd.DataFrame(effect_rows)

# Verify known frozen results.

def effect_value(benchmark, metric):
    rows = NSL11_WITHIN_BENCHMARK_EFFECTS.loc[
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Benchmark"]
            == benchmark
        )
        &
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Metric"]
            == metric
        )
    ]

    if len(rows) != 1:
        raise RuntimeError(
            "Expected one effect estimate."
        )

    return float(
        rows.iloc[0]["Within-Benchmark Change"]
    )

if not np.isclose(
    effect_value("CIC-IDS2017", "Macro F1"),
    0.018702,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro F1 effect verification failed."
    )

for metric in metric_names:
    if not np.isclose(
        effect_value("NSL-KDD", metric),
        0.0,
        atol=2e-9,
        rtol=0
    ):
        raise RuntimeError(
            f"Unexpected NSL effect: {metric}"
        )

# Verify paired correctness evidence.

cic_statistical_summary = ns11_load_json(
    CIC_STATS_DIR / "RF08_statistical_summary.json"
)

nsl_agreement = ns11_load_json(
    NSL10_DIR / "NSL10_prediction_agreement.json"
)

mcnemar_p = float(
    cic_statistical_summary["mcnemar"]["p_value"]
)

if not np.isclose(
    mcnemar_p,
    0.5384046,
    atol=1e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC McNemar result mismatch."
    )

if (
    nsl_agreement["validation_prediction_disagreements"]
    != 0
    or nsl_agreement[
        "official_test_prediction_disagreements"
    ] != 0
):
    raise RuntimeError(
        "NSL prediction agreement differs from Notebook 10."
    )

print("\nVERIFIED MODEL PERFORMANCE")

print(
    NSL11_MODEL_METRICS[
        [
            "Benchmark", "Model", "Classes",
            "Test Records", "Accuracy",
            "Macro Recall", "Macro F1", "MCC"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nWITHIN-BENCHMARK EFFECTS")

for benchmark in experiments:

    subset = NSL11_WITHIN_BENCHMARK_EFFECTS.loc[
        NSL11_WITHIN_BENCHMARK_EFFECTS["Benchmark"]
        == benchmark
    ]

    print("\n" + benchmark)

    print(
        subset[
            [
                "Metric",
                "Baseline Value",
                "Comparison Value",
                "Within-Benchmark Change"
            ]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:+.6f}"
        )
    )

print("\nPAIRED PREDICTION VERIFICATION")
print("CIC McNemar p-value:", f"{mcnemar_p:.7f}")
print(
    "NSL validation disagreements:",
    nsl_agreement["validation_prediction_disagreements"]
)
print(
    "NSL test disagreements:",
    nsl_agreement[
        "official_test_prediction_disagreements"
    ]
)

NSL11_METRICS_VERIFIED = True

print("\nCELL 02: PASSED")
print("Within-benchmark changes: VERIFIED")
print("Cross-benchmark raw score equivalence assumed: NO")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — WITHIN-BENCHMARK PERFORMANCE COMPARISON

VERIFIED MODEL PERFORMANCE
  Benchmark     Model  Classes  Test Records  Accuracy  Macro Recall  Macro F1      MCC
CIC-IDS2017  RF-B-CIC       15        378119  0.998606      0.879297  0.893851 0.995364
CIC-IDS2017  RF-E-CIC       15        378119  0.998633      0.927545  0.912553 0.995457
    NSL-KDD  RF-B-NSL        5         22544  0.758384      0.493121  0.504649 0.650888
    NSL-KDD RF-H2-NSL        5         22544  0.758384      0.493121  0.504649 0.650888

WITHIN-BENCHMARK EFFECTS

CIC-IDS2017
           Metric  Baseline Value  Comparison Value  Within-Benchmark Change
         Accuracy       +0.998606         +0.998633                +0.000026
  Macro Precision       +0.935485         +0.901210                -0.034275
     Macro Recall       +0.879297         +0.927545                +0.048248
         Macro F1       +0.893851         +0.912553                +0.018702
      Weighted F1       +0.998570         +0.998638       

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 03 — CLASS-LEVEL GENERALISATION AUDIT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 03 — CLASS-LEVEL GENERALISATION AUDIT")
print("=" * 72)

if not globals().get("NSL11_METRICS_VERIFIED", False):
    raise RuntimeError("Cell 02 must pass first.")

NSL11_CLASS_AUDIT_VERIFIED = False

NSL11_CIC_CLASS_RESULTS = pd.read_csv(
    CIC_STATS_DIR / "RF08_per_class_comparison.csv"
)

NSL11_NSL_VALIDATION_CLASSES = pd.read_csv(
    NSL10_DIR
    / "NSL10_validation_class_comparison.csv"
)

NSL11_NSL_TEST_CLASSES = pd.read_csv(
    NSL10_DIR
    / "NSL10_official_test_class_comparison.csv"
)

NSL11_NSL_VALIDATION_METRICS = pd.read_csv(
    NSL10_DIR
    / "NSL10_validation_model_comparison.csv"
).set_index("Metric")

NSL11_NSL_TEST_METRICS = pd.read_csv(
    NSL10_DIR
    / "NSL10_official_test_model_comparison.csv"
).set_index("Metric")

expected_nsl_classes = [
    "Normal", "DoS", "Probe", "R2L", "U2R"
]

# Verify class labels, supports and summary metrics.

for name, frame, expected_total in [
    ("CIC Test", NSL11_CIC_CLASS_RESULTS, 378119),
    ("NSL Validation", NSL11_NSL_VALIDATION_CLASSES, 18896),
    ("NSL Test", NSL11_NSL_TEST_CLASSES, 22544)
]:

    if frame["Class"].duplicated().any():
        raise RuntimeError(
            f"{name}: duplicate class names."
        )

    if int(frame["Support"].sum()) != expected_total:
        raise RuntimeError(
            f"{name}: class support mismatch."
        )

if len(NSL11_CIC_CLASS_RESULTS) != 15:
    raise RuntimeError(
        "Expected 15 CIC classes."
    )

for name, frame in [
    ("NSL Validation", NSL11_NSL_VALIDATION_CLASSES),
    ("NSL Test", NSL11_NSL_TEST_CLASSES)
]:

    if frame["Class"].tolist() != expected_nsl_classes:
        raise RuntimeError(
            f"{name}: class order mismatch."
        )

# Verify per-class F1 means against aggregate metrics.

for partition_name, frame, metric_table in [
    (
        "Validation",
        NSL11_NSL_VALIDATION_CLASSES,
        NSL11_NSL_VALIDATION_METRICS
    ),
    (
        "Official Test",
        NSL11_NSL_TEST_CLASSES,
        NSL11_NSL_TEST_METRICS
    )
]:

    for metric_column, model_column in [
        ("Baseline F1", "RF-B-NSL"),
        ("Tuned F1", "RF-H2-NSL")
    ]:

        macro_f1 = float(
            frame[metric_column].mean()
        )

        archived_macro_f1 = float(
            metric_table.loc["Macro F1", model_column]
        )

        if not np.isclose(
            macro_f1,
            archived_macro_f1,
            atol=3e-9,
            rtol=0
        ):
            raise RuntimeError(
                f"{partition_name}: Macro F1 mismatch "
                f"for {model_column}."
            )

# Calculate validation-to-test changes independently.

shift_records = []

for model_name in ["RF-B-NSL", "RF-H2-NSL"]:

    for metric in [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Balanced Accuracy",
        "MCC"
    ]:

        validation_value = float(
            NSL11_NSL_VALIDATION_METRICS.loc[
                metric, model_name
            ]
        )

        test_value = float(
            NSL11_NSL_TEST_METRICS.loc[
                metric, model_name
            ]
        )

        shift_records.append({
            "Model": model_name,
            "Metric": metric,
            "Validation": validation_value,
            "Official Test": test_value,
            "Test Minus Validation": (
                test_value - validation_value
            )
        })

NSL11_NSL_GENERALISATION_GAP = pd.DataFrame(
    shift_records
)

class_gap_records = []

validation_classes = (
    NSL11_NSL_VALIDATION_CLASSES.set_index("Class")
)

test_classes = (
    NSL11_NSL_TEST_CLASSES.set_index("Class")
)

for class_name in expected_nsl_classes:

    validation_row = validation_classes.loc[class_name]
    test_row = test_classes.loc[class_name]

    for model_name, prefix in [
        ("RF-B-NSL", "Baseline"),
        ("RF-H2-NSL", "Tuned")
    ]:

        val_f1 = float(
            validation_row[f"{prefix} F1"]
        )

        test_f1 = float(
            test_row[f"{prefix} F1"]
        )

        val_recall = (
            val_f1
            if False
            else None
        )

        class_gap_records.append({
            "Model": model_name,
            "Class": class_name,
            "Validation Support": int(
                validation_row["Support"]
            ),
            "Official Test Support": int(
                test_row["Support"]
            ),
            "Validation F1": val_f1,
            "Official Test F1": test_f1,
            "F1 Change": test_f1 - val_f1
        })

NSL11_NSL_CLASS_GENERALISATION = pd.DataFrame(
    class_gap_records
)

# Verify the 17 attack types absent from official training.

attack_inventory = pd.read_csv(
    NSL09_DIR / "NSL09_attack_inventory.csv"
)

if len(attack_inventory) != 40:
    raise RuntimeError(
        "Expected 40 NSL attack labels."
    )

test_only_mask = (
    attack_inventory["Training Count"].eq(0)
    & attack_inventory["Testing Count"].gt(0)
)

NSL11_TEST_ONLY_ATTACKS = (
    attack_inventory.loc[test_only_mask]
    .copy()
    .sort_values("Attack")
    .reset_index(drop=True)
)

if len(NSL11_TEST_ONLY_ATTACKS) != 17:
    raise RuntimeError(
        "Expected 17 NSL test-only attack types."
    )

test_only_records = int(
    NSL11_TEST_ONLY_ATTACKS["Testing Count"].sum()
)

if test_only_records > 22544:
    raise RuntimeError(
        "Test-only attack support exceeds dataset size."
    )

# Verify separate CIC class-level findings.

cic_macro_baseline = float(
    NSL11_CIC_CLASS_RESULTS["Baseline F1"].mean()
)

cic_macro_enhanced = float(
    NSL11_CIC_CLASS_RESULTS["Enhanced F1"].mean()
)

if not np.isclose(
    cic_macro_baseline, 0.893851,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC baseline class-level F1 mismatch."
    )

if not np.isclose(
    cic_macro_enhanced, 0.912553,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC enhanced class-level F1 mismatch."
    )

# Report findings.

print("\nNSL VALIDATION-TO-TEST PERFORMANCE")

print(
    NSL11_NSL_GENERALISATION_GAP[
        NSL11_NSL_GENERALISATION_GAP["Model"]
        == "RF-B-NSL"
    ][
        [
            "Metric",
            "Validation",
            "Official Test",
            "Test Minus Validation"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nNSL PER-CLASS GENERALISATION")

print(
    NSL11_NSL_CLASS_GENERALISATION[
        NSL11_NSL_CLASS_GENERALISATION["Model"]
        == "RF-B-NSL"
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nNSL TEST-ONLY ATTACK AUDIT")
print(
    "Attack types absent from training:",
    len(NSL11_TEST_ONLY_ATTACKS)
)
print(
    "Official-test records from these attack types:",
    test_only_records
)

print(
    NSL11_TEST_ONLY_ATTACKS[
        ["Attack", "Family", "Testing Count"]
    ].to_string(index=False)
)

print("\nCIC PER-CLASS EFFECTS")

print(
    NSL11_CIC_CLASS_RESULTS[
        [
            "Class", "Support",
            "Baseline F1",
            "Enhanced F1",
            "F1 Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nCLASS-LEVEL AUDIT")
print("CIC classes: 15")
print("NSL classes: 5")
print("CIC test observations: 378119")
print("NSL official test observations: 22544")
print("Class-level aggregate reconciliation: PASS")
print("NSL train/test attack inventory: PASS")
print("Different benchmark label spaces: ACKNOWLEDGED")
print("Direct raw Macro F1 comparison: EXCLUDED")

NSL11_CLASS_AUDIT_VERIFIED = True

print("\nCELL 03: PASSED")
print("Source experiments modified: NO")
print("Model training performed: NO")
print("=" * 72)


CELL 03 — CLASS-LEVEL GENERALISATION AUDIT

NSL VALIDATION-TO-TEST PERFORMANCE
           Metric  Validation  Official Test  Test Minus Validation
         Accuracy    0.999312       0.758384              -0.240928
  Macro Precision    0.971182       0.766484              -0.204698
     Macro Recall    0.946560       0.493121              -0.453439
         Macro F1    0.958147       0.504649              -0.453498
      Weighted F1    0.999306       0.717539              -0.281767
Balanced Accuracy    0.946560       0.493121              -0.453439
              MCC    0.998799       0.650888              -0.347911

NSL PER-CLASS GENERALISATION
   Model  Class  Validation Support  Official Test Support  Validation F1  Official Test F1  F1 Change
RF-B-NSL Normal               10102                   9711       0.999357          0.783829  -0.215528
RF-B-NSL    DoS                6889                   7460       0.999855          0.880332  -0.119523
RF-B-NSL  Probe                1748   

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 04 — EFFECT DIRECTION AND CLASS-LEVEL TRADE-OFFS
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 04 — EFFECT DIRECTION AND CLASS-LEVEL TRADE-OFFS")
print("=" * 72)

if not globals().get("NSL11_CLASS_AUDIT_VERIFIED", False):
    raise RuntimeError("Cell 03 must pass first.")

NSL11_EFFECT_AUDIT_VERIFIED = False

TOLERANCE = 1e-8

def classify_effect(change):
    if change > TOLERANCE:
        return "Improvement"
    if change < -TOLERANCE:
        return "Deterioration"
    return "No observed change"

# Derive metric-effect directions.

NSL11_EFFECT_DIRECTIONS = (
    NSL11_WITHIN_BENCHMARK_EFFECTS.copy()
)

NSL11_EFFECT_DIRECTIONS["Direction"] = (
    NSL11_EFFECT_DIRECTIONS[
        "Within-Benchmark Change"
    ].apply(classify_effect)
)

if len(NSL11_EFFECT_DIRECTIONS) != 14:
    raise RuntimeError("Unexpected metric-effect record count.")

# Derive class-level effect directions.

class_sources = {
    "CIC-IDS2017": {
        "data": NSL11_CIC_CLASS_RESULTS,
        "baseline": "Baseline F1",
        "comparison": "Enhanced F1",
        "expected_classes": 15
    },
    "NSL-KDD": {
        "data": NSL11_NSL_TEST_CLASSES,
        "baseline": "Baseline F1",
        "comparison": "Tuned F1",
        "expected_classes": 5
    }
}

class_records = []

for benchmark, source in class_sources.items():

    frame = source["data"]

    if len(frame) != source["expected_classes"]:
        raise RuntimeError(
            f"{benchmark}: class count mismatch."
        )

    for _, row in frame.iterrows():

        baseline_f1 = float(row[source["baseline"]])
        comparison_f1 = float(row[source["comparison"]])

        change = comparison_f1 - baseline_f1

        class_records.append({
            "Benchmark": benchmark,
            "Class": str(row["Class"]),
            "Support": int(row["Support"]),
            "Baseline F1": baseline_f1,
            "Comparison F1": comparison_f1,
            "F1 Change": change,
            "Direction": classify_effect(change)
        })

NSL11_CLASS_EFFECTS = pd.DataFrame(class_records)

if len(NSL11_CLASS_EFFECTS) != 20:
    raise RuntimeError(
        "Expected 20 benchmark-specific classes."
    )

# Verify class-level changes against aggregate Macro F1.

for benchmark in class_sources:

    subset = NSL11_CLASS_EFFECTS.loc[
        NSL11_CLASS_EFFECTS["Benchmark"] == benchmark
    ]

    aggregate_record = NSL11_EFFECT_DIRECTIONS.loc[
        (
            NSL11_EFFECT_DIRECTIONS["Benchmark"]
            == benchmark
        )
        &
        (
            NSL11_EFFECT_DIRECTIONS["Metric"]
            == "Macro F1"
        )
    ]

    if len(aggregate_record) != 1:
        raise RuntimeError(
            f"{benchmark}: aggregate Macro F1 unavailable."
        )

    class_mean_change = float(
        subset["F1 Change"].mean()
    )

    aggregate_change = float(
        aggregate_record.iloc[0][
            "Within-Benchmark Change"
        ]
    )

    if not np.isclose(
        class_mean_change,
        aggregate_change,
        rtol=0,
        atol=2e-6
    ):
        raise RuntimeError(
            f"{benchmark}: class-level Macro F1 "
            "reconciliation failed."
        )

# Verify expected class-effect distributions.

cic_effects = NSL11_CLASS_EFFECTS.loc[
    NSL11_CLASS_EFFECTS["Benchmark"]
    == "CIC-IDS2017"
]

nsl_effects = NSL11_CLASS_EFFECTS.loc[
    NSL11_CLASS_EFFECTS["Benchmark"]
    == "NSL-KDD"
]

expected_cic_directions = {
    "Improvement": 8,
    "Deterioration": 3,
    "No observed change": 4
}

for direction, expected in expected_cic_directions.items():

    actual = int(
        (cic_effects["Direction"] == direction).sum()
    )

    if actual != expected:
        raise RuntimeError(
            f"CIC class-effect mismatch for {direction}."
        )

if not (
    nsl_effects["Direction"] == "No observed change"
).all():
    raise RuntimeError(
        "NSL class effects differ from frozen results."
    )

if not np.isclose(
    float(cic_effects["F1 Change"].mean()),
    0.018702,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro F1 improvement mismatch."
    )

if not np.isclose(
    float(nsl_effects["F1 Change"].mean()),
    0.0,
    atol=1e-9,
    rtol=0
):
    raise RuntimeError(
        "NSL Macro F1 effect mismatch."
    )

print("\nMETRIC EFFECT DIRECTIONS")

print(
    NSL11_EFFECT_DIRECTIONS[
        [
            "Benchmark",
            "Metric",
            "Within-Benchmark Change",
            "Direction"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:+.6f}"
    )
)

print("\nCLASS-LEVEL EFFECT SUMMARY")

for benchmark in class_sources:

    subset = NSL11_CLASS_EFFECTS.loc[
        NSL11_CLASS_EFFECTS["Benchmark"] == benchmark
    ]

    print("\n" + benchmark)

    for direction in [
        "Improvement",
        "Deterioration",
        "No observed change"
    ]:
        print(
            f"{direction}:",
            int((subset["Direction"] == direction).sum())
        )

    print(
        "Mean class F1 change:",
        f"{subset['F1 Change'].mean():+.6f}"
    )

print("\nCIC CLASS-LEVEL TRADE-OFFS")

print(
    cic_effects[
        [
            "Class",
            "Support",
            "Baseline F1",
            "Comparison F1",
            "F1 Change",
            "Direction"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nNSL CLASS-LEVEL EFFECTS")

print(
    nsl_effects[
        [
            "Class",
            "Support",
            "Baseline F1",
            "Comparison F1",
            "F1 Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

NSL11_EFFECT_AUDIT_VERIFIED = True

print("\nINTEGRITY SUMMARY")
print("Metric-effect records: 14")
print("Class-effect records: 20")
print("CIC class-level reconciliation: PASS")
print("NSL class-level reconciliation: PASS")
print("CIC positive and negative effects: PRESERVED")
print("NSL unchanged prediction outcome: PRESERVED")
print("CELL 04: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 04 — EFFECT DIRECTION AND CLASS-LEVEL TRADE-OFFS

METRIC EFFECT DIRECTIONS
  Benchmark            Metric  Within-Benchmark Change          Direction
CIC-IDS2017          Accuracy                +0.000026        Improvement
CIC-IDS2017   Macro Precision                -0.034275      Deterioration
CIC-IDS2017      Macro Recall                +0.048248        Improvement
CIC-IDS2017          Macro F1                +0.018702        Improvement
CIC-IDS2017       Weighted F1                +0.000067        Improvement
CIC-IDS2017 Balanced Accuracy                +0.048248        Improvement
CIC-IDS2017               MCC                +0.000093        Improvement
    NSL-KDD          Accuracy                +0.000000 No observed change
    NSL-KDD   Macro Precision                +0.000000 No observed change
    NSL-KDD      Macro Recall                +0.000000 No observed change
    NSL-KDD          Macro F1                +0.000000 No observed change
    NSL-KDD       Weighted F1   

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 05 — NSL DISTRIBUTION SHIFT AND ATTACK NOVELTY
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 05 — NSL DISTRIBUTION SHIFT AND ATTACK NOVELTY")
print("=" * 72)

if not globals().get("NSL11_EFFECT_AUDIT_VERIFIED", False):
    raise RuntimeError("Cell 04 must pass first.")

NSL11_SHIFT_AUDIT_VERIFIED = False

NSL11_ATTACK_INVENTORY = pd.read_csv(
    NSL09_DIR / "NSL09_attack_inventory.csv"
)

class_order = [
    "Normal", "DoS", "Probe", "R2L", "U2R"
]

if len(NSL11_ATTACK_INVENTORY) != 40:
    raise RuntimeError("Expected 40 attack names.")

# Calculate original official-partition class distributions.

family_distribution = (
    NSL11_ATTACK_INVENTORY.groupby("Family")[
        ["Training Count", "Testing Count"]
    ]
    .sum()
    .reindex(class_order, fill_value=0)
)

if not np.array_equal(
    family_distribution["Training Count"].to_numpy(),
    [67343, 45927, 11656, 995, 52]
):
    raise RuntimeError(
        "Official training class distribution mismatch."
    )

if not np.array_equal(
    family_distribution["Testing Count"].to_numpy(),
    [9711, 7460, 2421, 2885, 67]
):
    raise RuntimeError(
        "Official test class distribution mismatch."
    )

training_total = int(
    family_distribution["Training Count"].sum()
)

test_total = int(
    family_distribution["Testing Count"].sum()
)

if training_total != 125973 or test_total != 22544:
    raise RuntimeError("Official partition totals differ.")

NSL11_DISTRIBUTION_SHIFT = family_distribution.reset_index()

NSL11_DISTRIBUTION_SHIFT[
    "Training Proportion"
] = (
    NSL11_DISTRIBUTION_SHIFT["Training Count"]
    / training_total
)

NSL11_DISTRIBUTION_SHIFT[
    "Test Proportion"
] = (
    NSL11_DISTRIBUTION_SHIFT["Testing Count"]
    / test_total
)

NSL11_DISTRIBUTION_SHIFT[
    "Change Percentage Points"
] = 100 * (
    NSL11_DISTRIBUTION_SHIFT["Test Proportion"]
    - NSL11_DISTRIBUTION_SHIFT["Training Proportion"]
)

# Reload the original official test labels.

source_record = NSL11_NSL09_MANIFEST[
    "source_files"
]["official_test"]

raw_test_path = Path(source_record["path"])

if ns11_sha256(raw_test_path) != source_record["sha256"]:
    raise RuntimeError(
        "Official test source fingerprint mismatch."
    )

raw_attack_names = (
    pd.read_csv(
        raw_test_path,
        header=None,
        usecols=[41],
        dtype=str
    )
    .iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.rstrip(".")
)

if len(raw_attack_names) != test_total:
    raise RuntimeError(
        "Raw official test observation count mismatch."
    )

class_mapping = json.loads(
    (
        PROJECT_ROOT
        / "data/NSL_KDD/processed/nslkdd_class_mapping.json"
    ).read_text(encoding="utf-8")
)

attack_to_family = class_mapping["attack_to_family"]
class_to_index = class_mapping["class_to_index"]

mapped_family = raw_attack_names.map(
    attack_to_family
)

if mapped_family.isna().any():
    raise RuntimeError(
        "Unmapped original test attack names."
    )

mapped_indices = (
    mapped_family.map(class_to_index)
    .to_numpy(dtype=np.int32)
)

processed_test_labels = (
    pd.read_parquet(
        PROJECT_ROOT
        / "data/NSL_KDD/processed/nslkdd_test.parquet",
        columns=["Label"]
    )["Label"]
    .to_numpy(dtype=np.int32)
)

if not np.array_equal(
    mapped_indices,
    processed_test_labels
):
    raise RuntimeError(
        "Original and processed test row alignment differs."
    )

# Reload independently saved test predictions.

baseline_predictions = np.load(
    NSL10_DIR
    / "RF_B_NSL/RF_B_NSL_test_predictions.npy",
    allow_pickle=False
).astype(np.int32)

tuned_predictions = np.load(
    NSL10_DIR
    / "RF_H2_NSL/RF_H2_NSL_test_predictions.npy",
    allow_pickle=False
).astype(np.int32)

if not (
    baseline_predictions.shape
    == tuned_predictions.shape
    == processed_test_labels.shape
    == (22544,)
):
    raise RuntimeError(
        "Saved NSL prediction dimensions differ."
    )

if not np.array_equal(
    baseline_predictions,
    tuned_predictions
):
    raise RuntimeError(
        "Saved NSL model predictions differ."
    )

# Identify test observations with attack names absent
# from the official training attack inventory.

training_attack_names = set(
    NSL11_ATTACK_INVENTORY.loc[
        NSL11_ATTACK_INVENTORY["Training Count"] > 0,
        "Attack"
    ].astype(str)
)

test_only_attack_names = set(
    NSL11_TEST_ONLY_ATTACKS["Attack"].astype(str)
)

novel_mask = (
    raw_attack_names.isin(test_only_attack_names)
    .to_numpy(dtype=bool)
)

seen_mask = ~novel_mask

if len(test_only_attack_names) != 17:
    raise RuntimeError(
        "Unexpected count of test-only attack types."
    )

if int(novel_mask.sum()) != 3750:
    raise RuntimeError(
        "Expected 3,750 test-only attack observations."
    )

if not set(
    raw_attack_names[seen_mask]
).issubset(training_attack_names):
    raise RuntimeError(
        "Seen-attack cohort contains an unknown attack."
    )

if not set(
    raw_attack_names[novel_mask]
).issubset(test_only_attack_names):
    raise RuntimeError(
        "Test-only cohort contains a known attack."
    )

correct = (
    baseline_predictions == processed_test_labels
)

# Cohort-level classification correctness.

cohort_records = []

for cohort_name, mask in [
    ("Training-seen attack names", seen_mask),
    ("Test-only attack names", novel_mask)
]:

    support = int(mask.sum())

    cohort_records.append({
        "Cohort": cohort_name,
        "Support": support,
        "Correct": int(np.count_nonzero(correct & mask)),
        "Incorrect": int(
            np.count_nonzero(~correct & mask)
        ),
        "Accuracy": float(
            np.mean(correct[mask])
        )
    })

NSL11_ATTACK_COHORT_RESULTS = pd.DataFrame(
    cohort_records
)

if int(NSL11_ATTACK_COHORT_RESULTS["Support"].sum()) != 22544:
    raise RuntimeError("Attack cohort support mismatch.")

if int(NSL11_ATTACK_COHORT_RESULTS["Correct"].sum()) != 17097:
    raise RuntimeError(
        "Attack cohort correct counts differ from NSL test."
    )

# Per-family attack novelty and recall within each cohort.

family_records = []

for index, family in enumerate(class_order):

    family_mask = (
        processed_test_labels == index
    )

    for cohort_name, cohort_mask in [
        ("Training-seen", seen_mask),
        ("Test-only", novel_mask)
    ]:

        mask = family_mask & cohort_mask
        support = int(mask.sum())
        correct_count = int(
            np.count_nonzero(correct & mask)
        )

        family_records.append({
            "Family": family,
            "Cohort": cohort_name,
            "Support": support,
            "Correct": correct_count,
            "Recall Within Cohort": (
                correct_count / support
                if support > 0 else np.nan
            )
        })

NSL11_FAMILY_NOVELTY_RESULTS = pd.DataFrame(
    family_records
)

if int(NSL11_FAMILY_NOVELTY_RESULTS["Support"].sum()) != 22544:
    raise RuntimeError(
        "Family novelty cohort totals differ."
    )

print("\nNSL CLASS DISTRIBUTION SHIFT")

print(
    NSL11_DISTRIBUTION_SHIFT.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nTEST-ONLY ATTACK EXPOSURE")
print("Distinct test-only attack types:", len(test_only_attack_names))
print("Test-only attack records:", int(novel_mask.sum()))
print(
    "Share of official test (%):",
    f"{100 * novel_mask.mean():.3f}"
)
print("Training-seen attack records:", int(seen_mask.sum()))

print("\nATTACK NOVELTY COHORT PERFORMANCE")

print(
    NSL11_ATTACK_COHORT_RESULTS.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nPER-FAMILY NOVELTY PERFORMANCE")

print(
    NSL11_FAMILY_NOVELTY_RESULTS.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

NSL11_SHIFT_AUDIT_VERIFIED = True

print("\nINTEGRITY SUMMARY")
print("Original source fingerprint: PASS")
print("Raw-to-processed test row alignment: PASS")
print("Official training/test family distributions: PASS")
print("Test-only attack classification: PASS")
print("Saved model predictions: IDENTICAL")
print("Cohort support and correct-count totals: PASS")
print("CELL 05: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 05 — NSL DISTRIBUTION SHIFT AND ATTACK NOVELTY

NSL CLASS DISTRIBUTION SHIFT
Family  Training Count  Testing Count  Training Proportion  Test Proportion  Change Percentage Points
Normal           67343           9711             0.534583         0.430758                -10.382518
   DoS           45927           7460             0.364578         0.330908                 -3.366968
 Probe           11656           2421             0.092528         0.107390                  1.486223
   R2L             995           2885             0.007899         0.127972                 12.007345
   U2R              52             67             0.000413         0.002972                  0.255918

TEST-ONLY ATTACK EXPOSURE
Distinct test-only attack types: 17
Test-only attack records: 3750
Share of official test (%): 16.634
Training-seen attack records: 18794

ATTACK NOVELTY COHORT PERFORMANCE
                    Cohort  Support  Correct  Incorrect  Accuracy
Training-seen attack names    18794    1

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 06 — CONSOLIDATED ROBUSTNESS EVIDENCE
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 06 — CONSOLIDATED ROBUSTNESS EVIDENCE")
print("=" * 72)

if not globals().get("NSL11_SHIFT_AUDIT_VERIFIED", False):
    raise RuntimeError("Cell 05 must pass first.")

NSL11_SYNTHESIS_VERIFIED = False

def ns11_metric_change(benchmark, metric):

    selected = NSL11_WITHIN_BENCHMARK_EFFECTS.loc[
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Benchmark"]
            == benchmark
        )
        &
        (
            NSL11_WITHIN_BENCHMARK_EFFECTS["Metric"]
            == metric
        )
    ]

    if len(selected) != 1:
        raise RuntimeError(
            f"Missing metric: {benchmark}, {metric}"
        )

    return float(
        selected.iloc[0]["Within-Benchmark Change"]
    )

cic_macro_f1_change = ns11_metric_change(
    "CIC-IDS2017", "Macro F1"
)

cic_macro_recall_change = ns11_metric_change(
    "CIC-IDS2017", "Macro Recall"
)

cic_macro_precision_change = ns11_metric_change(
    "CIC-IDS2017", "Macro Precision"
)

nsl_macro_f1_change = ns11_metric_change(
    "NSL-KDD", "Macro F1"
)

nsl_gap_record = NSL11_NSL_GENERALISATION_GAP.loc[
    (
        NSL11_NSL_GENERALISATION_GAP["Model"]
        == "RF-B-NSL"
    )
    &
    (
        NSL11_NSL_GENERALISATION_GAP["Metric"]
        == "Macro F1"
    )
]

if len(nsl_gap_record) != 1:
    raise RuntimeError(
        "NSL validation-to-test Macro F1 unavailable."
    )

nsl_macro_f1_gap = float(
    nsl_gap_record.iloc[0]["Test Minus Validation"]
)

nsl_baseline_test_report = pd.read_csv(
    NSL10_DIR
    / "RF_B_NSL/RF_B_NSL_test_classification_report.csv",
    index_col=0
)

r2l_recall = float(
    nsl_baseline_test_report.loc["R2L", "recall"]
)

u2r_recall = float(
    nsl_baseline_test_report.loc["U2R", "recall"]
)

r2l_support = int(
    nsl_baseline_test_report.loc["R2L", "support"]
)

u2r_support = int(
    nsl_baseline_test_report.loc["U2R", "support"]
)

novelty_records = int(
    NSL11_ATTACK_COHORT_RESULTS.loc[
        NSL11_ATTACK_COHORT_RESULTS["Cohort"]
        == "Test-only attack names",
        "Support"
    ].iloc[0]
)

novelty_share = novelty_records / 22544

nsl_agreement = ns11_load_json(
    NSL10_DIR / "NSL10_prediction_agreement.json"
)

cic_mcnemar = ns11_load_json(
    CIC_STATS_DIR / "RF08_McNemar_results.json"
)

cic_mcnemar_p = float(
    cic_mcnemar["p_value_two_sided"]
)

if not np.isclose(
    cic_macro_f1_change, 0.018702,
    rtol=0, atol=2e-6
):
    raise RuntimeError("CIC Macro F1 result mismatch.")

if not np.isclose(
    nsl_macro_f1_change, 0.0,
    rtol=0, atol=1e-9
):
    raise RuntimeError("NSL Macro F1 result mismatch.")

if not np.isclose(
    nsl_macro_f1_gap, -0.453498,
    rtol=0, atol=2e-6
):
    raise RuntimeError(
        "NSL generalisation gap mismatch."
    )

if not np.isclose(
    r2l_recall, 0.054766,
    rtol=0, atol=1e-6
):
    raise RuntimeError("R2L recall mismatch.")

if not np.isclose(
    u2r_recall, 0.029851,
    rtol=0, atol=1e-6
):
    raise RuntimeError("U2R recall mismatch.")

if (r2l_support, u2r_support) != (2885, 67):
    raise RuntimeError(
        "Minority-class test support mismatch."
    )

if novelty_records != 3750:
    raise RuntimeError(
        "NSL test-only attack exposure mismatch."
    )

if (
    nsl_agreement[
        "official_test_prediction_disagreements"
    ] != 0
):
    raise RuntimeError(
        "NSL frozen-tuning predictions differ."
    )

# Consolidated evidence structure.

NSL11_ROBUSTNESS_RECORD = {
    "datasets": {
        "CIC-IDS2017": {
            "test_records": 378119,
            "classes": 15,
            "comparison": "RF-B-CIC versus RF-E-CIC",
            "intervention": (
                "Depth restriction and training-only SMOTE"
            ),
            "macro_f1_change": cic_macro_f1_change,
            "macro_recall_change": cic_macro_recall_change,
            "macro_precision_change": cic_macro_precision_change,
            "mcnemar_p_value": cic_mcnemar_p
        },
        "NSL-KDD": {
            "test_records": 22544,
            "classes": 5,
            "comparison": "RF-B-NSL versus RF-H2-NSL",
            "intervention": (
                "Frozen maximum-depth restriction only"
            ),
            "macro_f1_change": nsl_macro_f1_change,
            "validation_to_test_macro_f1_change":
                nsl_macro_f1_gap,
            "test_prediction_disagreements": 0,
            "test_only_attack_types": 17,
            "test_only_attack_records": novelty_records,
            "test_only_attack_share": novelty_share,
            "r2l_test_recall": r2l_recall,
            "u2r_test_recall": u2r_recall
        }
    },
    "comparison_scope": (
        "Within-benchmark changes under distinct "
        "experimental interventions"
    ),
    "cross_dataset_model_transfer_performed": False,
    "model_training_performed": False
}

evidence_rows = [
    {
        "Finding": "CIC class-balanced performance",
        "Evidence": (
            f"Macro F1 change {cic_macro_f1_change:+.6f}; "
            f"Macro Recall change {cic_macro_recall_change:+.6f}"
        ),
        "Interpretation": (
            "Observed improvement with class-level trade-offs"
        )
    },
    {
        "Finding": "CIC paired correctness",
        "Evidence": (
            f"McNemar p-value {cic_mcnemar_p:.7f}"
        ),
        "Interpretation": (
            "No statistically significant difference "
            "in overall correctness at alpha 0.05"
        )
    },
    {
        "Finding": "NSL depth restriction",
        "Evidence": (
            "Zero validation and official-test "
            "prediction disagreements"
        ),
        "Interpretation": (
            "No observed classification effect "
            "from the frozen depth restriction"
        )
    },
    {
        "Finding": "NSL generalisation gap",
        "Evidence": (
            f"Macro F1 change from validation to "
            f"official test {nsl_macro_f1_gap:+.6f}"
        ),
        "Interpretation": (
            "Substantial performance deterioration "
            "between evaluation partitions"
        )
    },
    {
        "Finding": "NSL attack-type novelty",
        "Evidence": (
            f"{novelty_records} test records "
            f"({100 * novelty_share:.3f}%) "
            "from 17 training-unseen attack names"
        ),
        "Interpretation": (
            "Novel attack names and altered class "
            "composition are present in the official test"
        )
    },
    {
        "Finding": "NSL minority attack detection",
        "Evidence": (
            f"R2L recall {r2l_recall:.6f} "
            f"(n={r2l_support}); "
            f"U2R recall {u2r_recall:.6f} "
            f"(n={u2r_support})"
        ),
        "Interpretation": (
            "Weak official-test detection for R2L and U2R"
        )
    }
]

NSL11_EVIDENCE_TABLE = pd.DataFrame(evidence_rows)

NSL11_METHOD_LIMITATIONS = [
    (
        "CIC-IDS2017 has 15 target classes and NSL-KDD "
        "has five attack families; raw Macro F1 values "
        "represent different classification tasks."
    ),
    (
        "The CIC comparison includes class-imbalance "
        "handling, whereas the NSL comparison changes "
        "only the maximum-depth configuration."
    ),
    (
        "No fitted CIC model was transferred to NSL-KDD. "
        "The experiment tests independent methodological "
        "replication, not direct cross-dataset inference."
    ),
    (
        "NSL validation and official-test partitions differ "
        "in class proportions and observed attack names; "
        "the performance gap cannot be attributed to a "
        "single factor from these comparisons alone."
    ),
    (
        "Rare class estimates have limited support, "
        "including eight validation U2R observations "
        "and three CIC test SQL Injection observations."
    ),
    (
        "CIC paired statistical results concern overall "
        "classification correctness, not significance "
        "of Macro F1 differences; network-flow dependence "
        "may affect nominal significance."
    )
]

print("\nCONSOLIDATED ROBUSTNESS EVIDENCE")
print(
    NSL11_EVIDENCE_TABLE.to_string(index=False)
)

print("\nMETHODOLOGICAL LIMITATIONS")

for index, limitation in enumerate(
    NSL11_METHOD_LIMITATIONS,
    start=1
):
    print(f"{index}. {limitation}")

print("\nFINAL EVIDENCE CHECKS")
print("CIC experimental effect: VERIFIED")
print("CIC statistical conclusion: VERIFIED")
print("NSL independent replication effect: VERIFIED")
print("NSL generalisation gap: VERIFIED")
print("NSL test-only attack exposure: VERIFIED")
print("NSL minority-class recall: VERIFIED")
print("Benchmark label-space distinction: PRESERVED")
print("Direct model transfer claim: EXCLUDED")
print("Single-cause dataset-shift claim: EXCLUDED")

NSL11_SYNTHESIS_VERIFIED = True

print("\nCELL 06: PASSED")
print("Robustness evidence consolidation: COMPLETE")
print("Model training performed: NO")
print("=" * 72)


CELL 06 — CONSOLIDATED ROBUSTNESS EVIDENCE

CONSOLIDATED ROBUSTNESS EVIDENCE
                       Finding                                                         Evidence                                                                    Interpretation
CIC class-balanced performance         Macro F1 change +0.018702; Macro Recall change +0.048248                                  Observed improvement with class-level trade-offs
        CIC paired correctness                                        McNemar p-value 0.5384046      No statistically significant difference in overall correctness at alpha 0.05
         NSL depth restriction       Zero validation and official-test prediction disagreements               No observed classification effect from the frozen depth restriction
        NSL generalisation gap       Macro F1 change from validation to official test -0.453498               Substantial performance deterioration between evaluation partitions
       NSL attack-type novelty 37

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 07 — COMPARATIVE TABLE EXPORT
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 07 — COMPARATIVE TABLE EXPORT")
print("=" * 72)

required_stages = [
    "NSL11_INPUTS_VERIFIED",
    "NSL11_METRICS_VERIFIED",
    "NSL11_CLASS_AUDIT_VERIFIED",
    "NSL11_EFFECT_AUDIT_VERIFIED",
    "NSL11_SHIFT_AUDIT_VERIFIED",
    "NSL11_SYNTHESIS_VERIFIED"
]

for stage in required_stages:
    if not globals().get(stage, False):
        raise RuntimeError(f"Required stage incomplete: {stage}")

NSL11_TABLES_EXPORTED = False

NSL11_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

def ns11_write_once(path, content):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if not isinstance(content, bytes):
        raise TypeError("Expected byte content.")

    try:
        with path.open("xb") as handle:
            handle.write(content)

        return "CREATED"

    except FileExistsError:

        with path.open("rb") as handle:
            existing = handle.read()

        if hashlib.sha256(existing).digest() != (
            hashlib.sha256(content).digest()
        ):
            raise RuntimeError(
                f"Existing output differs: {path}"
            )

        return "ALREADY VERIFIED"

NSL11_TABLE_EXPORTS = {
    "NSL11_model_test_metrics.csv":
        NSL11_MODEL_METRICS,

    "NSL11_within_benchmark_effects.csv":
        NSL11_WITHIN_BENCHMARK_EFFECTS,

    "NSL11_metric_effect_directions.csv":
        NSL11_EFFECT_DIRECTIONS,

    "NSL11_class_effects.csv":
        NSL11_CLASS_EFFECTS,

    "NSL11_nsl_generalisation_gap.csv":
        NSL11_NSL_GENERALISATION_GAP,

    "NSL11_nsl_class_generalisation.csv":
        NSL11_NSL_CLASS_GENERALISATION,

    "NSL11_nsl_family_distribution_shift.csv":
        NSL11_DISTRIBUTION_SHIFT,

    "NSL11_nsl_novelty_cohort_performance.csv":
        NSL11_ATTACK_COHORT_RESULTS,

    "NSL11_nsl_family_novelty_performance.csv":
        NSL11_FAMILY_NOVELTY_RESULTS,

    "NSL11_nsl_test_only_attack_inventory.csv":
        NSL11_TEST_ONLY_ATTACKS,

    "NSL11_robustness_evidence.csv":
        NSL11_EVIDENCE_TABLE
}

EXPECTED_ROWS = {
    "NSL11_model_test_metrics.csv": 4,
    "NSL11_within_benchmark_effects.csv": 14,
    "NSL11_metric_effect_directions.csv": 14,
    "NSL11_class_effects.csv": 20,
    "NSL11_nsl_generalisation_gap.csv": 14,
    "NSL11_nsl_class_generalisation.csv": 10,
    "NSL11_nsl_family_distribution_shift.csv": 5,
    "NSL11_nsl_novelty_cohort_performance.csv": 2,
    "NSL11_nsl_family_novelty_performance.csv": 10,
    "NSL11_nsl_test_only_attack_inventory.csv": 17,
    "NSL11_robustness_evidence.csv": 6
}

if set(NSL11_TABLE_EXPORTS) != set(EXPECTED_ROWS):
    raise RuntimeError("Table export specification mismatch.")

# Reconcile key table totals.

if int(
    NSL11_CLASS_EFFECTS.loc[
        NSL11_CLASS_EFFECTS["Benchmark"] == "CIC-IDS2017",
        "Support"
    ].sum()
) != 378119:
    raise RuntimeError("CIC class supports differ.")

if int(
    NSL11_CLASS_EFFECTS.loc[
        NSL11_CLASS_EFFECTS["Benchmark"] == "NSL-KDD",
        "Support"
    ].sum()
) != 22544:
    raise RuntimeError("NSL class supports differ.")

if int(
    NSL11_ATTACK_COHORT_RESULTS["Support"].sum()
) != 22544:
    raise RuntimeError("NSL cohort supports differ.")

if int(
    NSL11_ATTACK_COHORT_RESULTS["Correct"].sum()
) != 17097:
    raise RuntimeError("NSL cohort correct counts differ.")

if int(
    NSL11_TEST_ONLY_ATTACKS["Testing Count"].sum()
) != 3750:
    raise RuntimeError("NSL unseen-attack total differs.")

if not np.isclose(
    NSL11_DISTRIBUTION_SHIFT[
        "Training Proportion"
    ].sum(),
    1.0,
    rtol=0,
    atol=1e-12
):
    raise RuntimeError("Training proportions do not sum to one.")

if not np.isclose(
    NSL11_DISTRIBUTION_SHIFT[
        "Test Proportion"
    ].sum(),
    1.0,
    rtol=0,
    atol=1e-12
):
    raise RuntimeError("Testing proportions do not sum to one.")

NSL11_EXPORTED_TABLE_PATHS = []

print("\nCOMPARATIVE TABLE EXPORTS")

for filename, dataframe in NSL11_TABLE_EXPORTS.items():

    if not isinstance(dataframe, pd.DataFrame):
        raise RuntimeError(
            f"Invalid table object: {filename}"
        )

    if dataframe.empty:
        raise RuntimeError(
            f"Empty export table: {filename}"
        )

    if dataframe.columns.duplicated().any():
        raise RuntimeError(
            f"Duplicate columns: {filename}"
        )

    if len(dataframe) != EXPECTED_ROWS[filename]:
        raise RuntimeError(
            f"{filename}: unexpected row count."
        )

    path = NSL11_RESULTS_DIR / filename

    content = dataframe.to_csv(
        index=False,
        float_format="%.9f",
        na_rep=""
    ).encode("utf-8")

    status = ns11_write_once(path, content)

    reloaded = pd.read_csv(path)

    if reloaded.shape != dataframe.shape:
        raise RuntimeError(
            f"{filename}: saved table dimensions differ."
        )

    if list(reloaded.columns) != list(dataframe.columns):
        raise RuntimeError(
            f"{filename}: saved column names differ."
        )

    NSL11_EXPORTED_TABLE_PATHS.append(path)

    print(
        f"{filename}: {status} "
        f"({len(dataframe)} rows)"
    )

if len(NSL11_EXPORTED_TABLE_PATHS) != 11:
    raise RuntimeError("Unexpected exported table count.")

NSL11_TABLES_EXPORTED = True

print("\nTABLE INTEGRITY")
print("Exported tables:", len(NSL11_EXPORTED_TABLE_PATHS))
print("Model metric records: 4")
print("Metric effect records: 14")
print("Class-level effect records: 20")
print("NSL test-only attack types: 17")
print("NSL cohort correct predictions: 17097")
print("Saved table dimensions: PASS")
print("Source result reconciliation: PASS")
print("Original experiment files overwritten: NO")

print("\nCELL 07: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 07 — COMPARATIVE TABLE EXPORT

COMPARATIVE TABLE EXPORTS
NSL11_model_test_metrics.csv: CREATED (4 rows)
NSL11_within_benchmark_effects.csv: CREATED (14 rows)
NSL11_metric_effect_directions.csv: CREATED (14 rows)
NSL11_class_effects.csv: CREATED (20 rows)
NSL11_nsl_generalisation_gap.csv: CREATED (14 rows)
NSL11_nsl_class_generalisation.csv: CREATED (10 rows)
NSL11_nsl_family_distribution_shift.csv: CREATED (5 rows)
NSL11_nsl_novelty_cohort_performance.csv: CREATED (2 rows)
NSL11_nsl_family_novelty_performance.csv: CREATED (10 rows)
NSL11_nsl_test_only_attack_inventory.csv: CREATED (17 rows)
NSL11_robustness_evidence.csv: CREATED (6 rows)

TABLE INTEGRITY
Exported tables: 11
Model metric records: 4
Metric effect records: 14
Class-level effect records: 20
NSL test-only attack types: 17
NSL cohort correct predictions: 17097
Saved table dimensions: PASS
Source result reconciliation: PASS
Original experiment files overwritten: NO

CELL 07: PASSED
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 08 — ROBUSTNESS FIGURE GENERATION
# ============================================================

from pathlib import Path
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 72)
print("CELL 08 — ROBUSTNESS FIGURE GENERATION")
print("=" * 72)

if not globals().get("NSL11_TABLES_EXPORTED", False):
    raise RuntimeError("Cell 07 must pass first.")

NSL11_FIGURES_EXPORTED = False

NSL11_FIGURE_PATHS = []

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

CIC_COLOR = "#2563A6"
NSL_COLOR = "#D48A32"
TEST_COLOR = "#B45353"
GRID_COLOR = "#D9E0E8"

def ns11_export_figure(filename, figure):

    path = NSL11_RESULTS_DIR / filename

    buffer = io.BytesIO()

    figure.savefig(
        buffer,
        format="png",
        dpi=200,
        bbox_inches="tight",
        metadata={"Software": "Matplotlib"}
    )

    plt.close(figure)

    content = buffer.getvalue()

    if not content.startswith(
        b"\x89PNG\r\n\x1a\n"
    ):
        raise RuntimeError(
            f"Invalid PNG content: {filename}"
        )

    status = ns11_write_once(path, content)

    if path.stat().st_size < 10000:
        raise RuntimeError(
            f"Unexpectedly small figure: {filename}"
        )

    NSL11_FIGURE_PATHS.append(path)

    print(f"{filename}: {status}")


# ------------------------------------------------------------
# Figure 1 — Within-benchmark metric changes
# ------------------------------------------------------------

selected_metrics = [
    "Macro Precision",
    "Macro Recall",
    "Macro F1"
]

effect_table = NSL11_WITHIN_BENCHMARK_EFFECTS

def ns11_effect_array(benchmark):

    values = []

    for metric in selected_metrics:

        row = effect_table.loc[
            (effect_table["Benchmark"] == benchmark)
            & (effect_table["Metric"] == metric)
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"Missing effect: {benchmark}, {metric}"
            )

        values.append(
            float(
                row.iloc[0]["Within-Benchmark Change"]
            )
        )

    return np.asarray(values)

cic_effects = ns11_effect_array("CIC-IDS2017")
nsl_effects = ns11_effect_array("NSL-KDD")

figure, axis = plt.subplots(figsize=(10, 6))

positions = np.arange(len(selected_metrics))
width = 0.34

axis.bar(
    positions - width / 2,
    cic_effects,
    width=width,
    color=CIC_COLOR,
    label="CIC-IDS2017 enhancement"
)

axis.bar(
    positions + width / 2,
    nsl_effects,
    width=width,
    color=NSL_COLOR,
    label="NSL-KDD frozen-depth replication"
)

axis.axhline(0, color="#333333", linewidth=1)

axis.set_xticks(positions)
axis.set_xticklabels(selected_metrics)

axis.set_ylabel("Change relative to baseline")
axis.set_title("Within-Benchmark Random Forest Effects")
axis.grid(axis="y", color=GRID_COLOR, alpha=0.7)
axis.set_axisbelow(True)
axis.legend(loc="upper left", fontsize=9)

for index, value in enumerate(cic_effects):

    axis.annotate(
        f"{value:+.4f}",
        (
            positions[index] - width / 2,
            value
        ),
        xytext=(0, 5 if value >= 0 else -15),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

figure.text(
    0.5,
    0.015,
    "Effects are within each benchmark; interventions and "
    "target label spaces differ.",
    ha="center",
    fontsize=9,
    color="#555555"
)

figure.tight_layout(rect=(0, 0.055, 1, 1))

ns11_export_figure(
    "NSL11_within_benchmark_metric_effects.png",
    figure
)


# ------------------------------------------------------------
# Figure 2 — NSL validation and official-test F1
# ------------------------------------------------------------

nsl_class_gap = (
    NSL11_NSL_CLASS_GENERALISATION.loc[
        NSL11_NSL_CLASS_GENERALISATION["Model"]
        == "RF-B-NSL"
    ]
    .set_index("Class")
    .loc[["Normal", "DoS", "Probe", "R2L", "U2R"]]
)

if len(nsl_class_gap) != 5:
    raise RuntimeError("Missing NSL class-level results.")

figure, axis = plt.subplots(figsize=(11, 6.5))

positions = np.arange(5)
height = 0.35

axis.barh(
    positions - height / 2,
    nsl_class_gap["Validation F1"],
    height=height,
    color=CIC_COLOR,
    label="Development validation"
)

axis.barh(
    positions + height / 2,
    nsl_class_gap["Official Test F1"],
    height=height,
    color=TEST_COLOR,
    label="Official test"
)

axis.set_yticks(positions)
axis.set_yticklabels(nsl_class_gap.index)
axis.invert_yaxis()
axis.set_xlim(0, 1.06)
axis.set_xlabel("F1-score")
axis.set_title("NSL-KDD: Class-Level Generalisation Gap")
axis.grid(axis="x", color=GRID_COLOR, alpha=0.7)
axis.set_axisbelow(True)
axis.legend(loc="lower right")

figure.text(
    0.5,
    0.015,
    "RF-B-NSL shown; RF-H2-NSL produced identical "
    "predictions on both partitions.",
    ha="center",
    fontsize=9,
    color="#555555"
)

figure.tight_layout(rect=(0, 0.06, 1, 1))

ns11_export_figure(
    "NSL11_nsl_validation_vs_test_class_f1.png",
    figure
)


# ------------------------------------------------------------
# Figure 3 — Official NSL family-distribution change
# ------------------------------------------------------------

family_distribution = (
    NSL11_DISTRIBUTION_SHIFT
    .set_index("Family")
    .loc[["Normal", "DoS", "Probe", "R2L", "U2R"]]
)

figure, axis = plt.subplots(figsize=(10, 6))

positions = np.arange(5)
width = 0.34

train_proportions = (
    family_distribution["Training Proportion"]
    .to_numpy(dtype=float) * 100
)

test_proportions = (
    family_distribution["Test Proportion"]
    .to_numpy(dtype=float) * 100
)

axis.bar(
    positions - width / 2,
    train_proportions,
    width=width,
    color=CIC_COLOR,
    label="Official training"
)

axis.bar(
    positions + width / 2,
    test_proportions,
    width=width,
    color=TEST_COLOR,
    label="Official test"
)

axis.set_xticks(positions)
axis.set_xticklabels(family_distribution.index)
axis.set_ylabel("Share of partition (%)")
axis.set_title("NSL-KDD: Attack-Family Distribution Shift")
axis.grid(axis="y", color=GRID_COLOR, alpha=0.7)
axis.set_axisbelow(True)
axis.legend()

figure.tight_layout()

ns11_export_figure(
    "NSL11_nsl_attack_family_distribution.png",
    figure
)


# ------------------------------------------------------------
# Figure 4 — Overall seen-versus-test-only accuracy
# ------------------------------------------------------------

cohorts = NSL11_ATTACK_COHORT_RESULTS.copy()

if set(cohorts["Cohort"]) != {
    "Training-seen attack names",
    "Test-only attack names"
}:
    raise RuntimeError("Unexpected attack cohorts.")

cohorts = cohorts.set_index("Cohort").loc[
    [
        "Training-seen attack names",
        "Test-only attack names"
    ]
]

figure, axis = plt.subplots(figsize=(9, 6))

values = cohorts["Accuracy"].to_numpy(dtype=float)
supports = cohorts["Support"].to_numpy(dtype=int)

bars = axis.bar(
    [0, 1],
    values,
    color=[CIC_COLOR, TEST_COLOR],
    width=0.55
)

axis.set_xticks([0, 1])
axis.set_xticklabels([
    "Training-seen\nattack names",
    "Test-only\nattack names"
])

axis.set_ylim(0, 1.05)
axis.set_ylabel("Classification accuracy")
axis.set_title("NSL-KDD: Test Accuracy by Attack-Name Novelty")
axis.grid(axis="y", color=GRID_COLOR, alpha=0.7)
axis.set_axisbelow(True)

for bar, value, support in zip(bars, values, supports):

    axis.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.2%}\nn={support:,}",
        ha="center",
        va="bottom",
        fontsize=10
    )

figure.text(
    0.5,
    0.015,
    "Descriptive cohorts with different class compositions; "
    "not a causal estimate.",
    ha="center",
    fontsize=9,
    color="#555555"
)

figure.tight_layout(rect=(0, 0.06, 1, 1))

ns11_export_figure(
    "NSL11_nsl_attack_novelty_accuracy.png",
    figure
)


# ------------------------------------------------------------
# Figure 5 — Attack novelty by class family
# ------------------------------------------------------------

novelty = NSL11_FAMILY_NOVELTY_RESULTS.copy()

families = ["Normal", "DoS", "Probe", "R2L", "U2R"]

seen_rows = (
    novelty.loc[novelty["Cohort"] == "Training-seen"]
    .set_index("Family")
    .loc[families]
)

new_rows = (
    novelty.loc[novelty["Cohort"] == "Test-only"]
    .set_index("Family")
    .loc[families]
)

seen_values = seen_rows[
    "Recall Within Cohort"
].to_numpy(dtype=float)

new_values = new_rows[
    "Recall Within Cohort"
].to_numpy(dtype=float)

new_support = new_rows[
    "Support"
].to_numpy(dtype=int)

figure, axis = plt.subplots(figsize=(11, 6))

positions = np.arange(5)
width = 0.34

axis.bar(
    positions - width / 2,
    seen_values,
    width=width,
    color=CIC_COLOR,
    label="Training-seen attack names"
)

available = new_support > 0

axis.bar(
    positions[available] + width / 2,
    new_values[available],
    width=width,
    color=TEST_COLOR,
    label="Test-only attack names"
)

for position in positions[~available]:

    axis.text(
        position + width / 2,
        0.04,
        "N/A",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#555555"
    )

axis.set_xticks(positions)
axis.set_xticklabels(families)
axis.set_ylim(0, 1.1)
axis.set_ylabel("Recall within cohort")
axis.set_title("NSL-KDD: Class Recall by Attack-Name Novelty")
axis.grid(axis="y", color=GRID_COLOR, alpha=0.7)
axis.set_axisbelow(True)
axis.legend(loc="upper right", fontsize=9)

figure.text(
    0.5,
    0.015,
    "N/A indicates no test-only Normal observations. "
    "Cohort sample sizes differ.",
    ha="center",
    fontsize=9,
    color="#555555"
)

figure.tight_layout(rect=(0, 0.06, 1, 1))

ns11_export_figure(
    "NSL11_nsl_family_novelty_recall.png",
    figure
)


# ------------------------------------------------------------
# Final figure verification
# ------------------------------------------------------------

if len(NSL11_FIGURE_PATHS) != 5:
    raise RuntimeError("Expected five exported figures.")

if len(set(NSL11_FIGURE_PATHS)) != 5:
    raise RuntimeError("Duplicate figure destinations.")

for path in NSL11_FIGURE_PATHS:

    if not path.is_file():
        raise FileNotFoundError(path)

    with path.open("rb") as handle:
        header = handle.read(8)

    if header != b"\x89PNG\r\n\x1a\n":
        raise RuntimeError(
            f"Invalid PNG signature: {path.name}"
        )

NSL11_FIGURES_EXPORTED = True

print("\nFIGURE INTEGRITY")
print("Exported figures:", len(NSL11_FIGURE_PATHS))
print("PNG format verification: PASS")
print("Within-benchmark effect presentation: PASS")
print("NSL class-level generalisation: PASS")
print("NSL class-distribution presentation: PASS")
print("Attack-novelty cohort presentation: PASS")
print("Source experiments modified: NO")

print("\nCELL 08: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 08 — ROBUSTNESS FIGURE GENERATION
NSL11_within_benchmark_metric_effects.png: CREATED
NSL11_nsl_validation_vs_test_class_f1.png: CREATED
NSL11_nsl_attack_family_distribution.png: CREATED
NSL11_nsl_attack_novelty_accuracy.png: CREATED
NSL11_nsl_family_novelty_recall.png: CREATED

FIGURE INTEGRITY
Exported figures: 5
PNG format verification: PASS
Within-benchmark effect presentation: PASS
NSL class-level generalisation: PASS
NSL class-distribution presentation: PASS
Attack-novelty cohort presentation: PASS
Source experiments modified: NO

CELL 08: PASSED
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 09 — INTERPRETATION RECORD AND EXPORT AUDIT
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 09 — INTERPRETATION RECORD AND EXPORT AUDIT")
print("=" * 72)

if not globals().get("NSL11_TABLES_EXPORTED", False):
    raise RuntimeError("Cell 07 must pass first.")

if not globals().get("NSL11_FIGURES_EXPORTED", False):
    raise RuntimeError("Cell 08 must pass first.")

NSL11_INTERPRETATION_VERIFIED = False

# ------------------------------------------------------------
# 1. Reconcile central experimental results
# ------------------------------------------------------------

model_metrics = (
    NSL11_MODEL_METRICS.set_index(
        ["Benchmark", "Model"]
    )
)

cic_baseline_f1 = float(
    model_metrics.loc[
        ("CIC-IDS2017", "RF-B-CIC"), "Macro F1"
    ]
)

cic_enhanced_f1 = float(
    model_metrics.loc[
        ("CIC-IDS2017", "RF-E-CIC"), "Macro F1"
    ]
)

cic_f1_change = cic_enhanced_f1 - cic_baseline_f1

cic_recall_change = ns11_metric_change(
    "CIC-IDS2017", "Macro Recall"
)

cic_precision_change = ns11_metric_change(
    "CIC-IDS2017", "Macro Precision"
)

nsl_baseline_f1 = float(
    model_metrics.loc[
        ("NSL-KDD", "RF-B-NSL"), "Macro F1"
    ]
)

nsl_tuned_f1 = float(
    model_metrics.loc[
        ("NSL-KDD", "RF-H2-NSL"), "Macro F1"
    ]
)

nsl_f1_change = nsl_tuned_f1 - nsl_baseline_f1

nsl_validation_f1 = float(
    NSL11_NSL_VALIDATION_METRICS.loc[
        "Macro F1", "RF-B-NSL"
    ]
)

nsl_test_f1 = float(
    NSL11_NSL_TEST_METRICS.loc[
        "Macro F1", "RF-B-NSL"
    ]
)

nsl_generalisation_gap = nsl_test_f1 - nsl_validation_f1

nsl_disagreements = ns11_load_json(
    NSL10_DIR / "NSL10_prediction_agreement.json"
)

cic_statistical_result = ns11_load_json(
    CIC_STATS_DIR / "RF08_McNemar_results.json"
)

cic_mcnemar_p = float(
    cic_statistical_result["p_value_two_sided"]
)

if not np.isclose(
    cic_f1_change, 0.018702,
    rtol=0, atol=2e-6
):
    raise RuntimeError("CIC Macro F1 change mismatch.")

if not np.isclose(
    nsl_f1_change, 0.0,
    rtol=0, atol=1e-9
):
    raise RuntimeError("NSL Macro F1 change mismatch.")

if not np.isclose(
    nsl_generalisation_gap, -0.453498,
    rtol=0, atol=2e-6
):
    raise RuntimeError(
        "NSL validation-to-test gap mismatch."
    )

if not np.isclose(
    cic_mcnemar_p, 0.5384046,
    rtol=0, atol=1e-6
):
    raise RuntimeError("CIC McNemar result mismatch.")

if (
    nsl_disagreements["validation_prediction_disagreements"]
    != 0
    or
    nsl_disagreements[
        "official_test_prediction_disagreements"
    ] != 0
):
    raise RuntimeError(
        "NSL model prediction agreement mismatch."
    )

# ------------------------------------------------------------
# 2. Reconcile attack novelty and error counts
# ------------------------------------------------------------

cohorts = (
    NSL11_ATTACK_COHORT_RESULTS.set_index("Cohort")
)

seen = cohorts.loc["Training-seen attack names"]
novel = cohorts.loc["Test-only attack names"]

seen_count = int(seen["Support"])
novel_count = int(novel["Support"])

seen_correct = int(seen["Correct"])
novel_correct = int(novel["Correct"])

seen_errors = int(seen["Incorrect"])
novel_errors = int(novel["Incorrect"])

total_errors = seen_errors + novel_errors

if (seen_count, novel_count) != (18794, 3750):
    raise RuntimeError("Attack cohort supports differ.")

if (seen_correct, novel_correct) != (16398, 699):
    raise RuntimeError("Attack cohort correct counts differ.")

if total_errors != 5447:
    raise RuntimeError(
        "Official-test total error count mismatch."
    )

novel_error_share = novel_errors / total_errors

if not np.isclose(
    novel_error_share,
    3051 / 5447,
    rtol=0, atol=1e-12
):
    raise RuntimeError(
        "Novel-attack error contribution mismatch."
    )

# ------------------------------------------------------------
# 3. Verify minority-family performance
# ------------------------------------------------------------

nsl_report_path = (
    NSL10_DIR
    / "RF_B_NSL/RF_B_NSL_test_classification_report.csv"
)

nsl_report = pd.read_csv(
    nsl_report_path,
    index_col=0
)

r2l_recall = float(
    nsl_report.loc["R2L", "recall"]
)

u2r_recall = float(
    nsl_report.loc["U2R", "recall"]
)

if not np.isclose(
    r2l_recall, 0.054766,
    rtol=0, atol=1e-6
):
    raise RuntimeError("R2L recall mismatch.")

if not np.isclose(
    u2r_recall, 0.029851,
    rtol=0, atol=1e-6
):
    raise RuntimeError("U2R recall mismatch.")

# ------------------------------------------------------------
# 4. Construct interpretation record
# ------------------------------------------------------------

NSL11_INTERPRETATION_RECORD = {
    "notebook": "11_Cross_Benchmark_Robustness_Analysis",
    "analysis_type": "cross_benchmark_evidence_synthesis",

    "cic_ids2017": {
        "classes": 15,
        "test_records": 378119,
        "baseline_model": "RF-B-CIC",
        "enhanced_model": "RF-E-CIC",
        "baseline_macro_f1": cic_baseline_f1,
        "enhanced_macro_f1": cic_enhanced_f1,
        "macro_f1_change": cic_f1_change,
        "macro_recall_change": cic_recall_change,
        "macro_precision_change": cic_precision_change,
        "mcnemar_p_value": cic_mcnemar_p,
        "mcnemar_scope": (
            "Paired overall classification correctness"
        ),
        "macro_f1_significance_test_performed": False,
        "intervention": (
            "Depth restriction and training-only SMOTE"
        )
    },

    "nsl_kdd": {
        "classes": 5,
        "official_test_records": 22544,
        "baseline_model": "RF-B-NSL",
        "tuned_model": "RF-H2-NSL",
        "baseline_macro_f1": nsl_baseline_f1,
        "tuned_macro_f1": nsl_tuned_f1,
        "macro_f1_change": nsl_f1_change,
        "validation_macro_f1": nsl_validation_f1,
        "official_test_macro_f1": nsl_test_f1,
        "validation_to_test_macro_f1_change":
            nsl_generalisation_gap,
        "validation_prediction_disagreements": 0,
        "test_prediction_disagreements": 0,
        "test_only_attack_type_count": 17,
        "test_only_attack_records": novel_count,
        "test_only_attack_record_share":
            novel_count / 22544,
        "seen_attack_name_accuracy":
            seen_correct / seen_count,
        "test_only_attack_name_accuracy":
            novel_correct / novel_count,
        "test_only_attack_name_errors": novel_errors,
        "total_test_errors": total_errors,
        "share_of_test_errors_from_test_only_names":
            novel_error_share,
        "r2l_test_recall": r2l_recall,
        "u2r_test_recall": u2r_recall,
        "intervention": (
            "Frozen maximum-depth restriction only"
        )
    },

    "interpretation_boundaries": [
        "Within-benchmark effects are the primary comparisons.",
        "CIC and NSL use different class definitions and class counts.",
        "NSL independently replicates the frozen depth setting, not the full CIC enhancement.",
        "Identical NSL predictions do not prove identical tree structures.",
        "Attack-name novelty cohorts differ in class composition.",
        "Dataset-shift effects cannot be attributed to one factor alone.",
        "CIC McNemar results do not measure Macro F1 significance.",
        "Small minority-class supports limit estimate stability.",
        "Related network-flow observations may not be independent.",
        "Repeated use of the CIC test partition limits confirmatory interpretation."
    ],

    "cross_dataset_model_transfer_performed": False,
    "new_hyperparameter_search_performed": False,
    "model_training_performed": False
}

# ------------------------------------------------------------
# 5. Generate technical findings summary
# ------------------------------------------------------------

NSL11_FINDINGS_TEXT = f"""CROSS-BENCHMARK ROBUSTNESS ANALYSIS
===================================

1. CIC-IDS2017

Baseline model: RF-B-CIC
Comparison model: RF-E-CIC
Test observations: 378119
Target classes: 15

Baseline Macro F1: {cic_baseline_f1:.6f}
Enhanced Macro F1: {cic_enhanced_f1:.6f}
Macro F1 change: {cic_f1_change:+.6f}
Macro Recall change: {cic_recall_change:+.6f}
Macro Precision change: {cic_precision_change:+.6f}

The enhanced model achieved higher observed Macro F1 and
Macro Recall, accompanied by lower Macro Precision.
Class-level improvements were not uniform.

McNemar two-sided p-value: {cic_mcnemar_p:.7f}
At alpha = 0.05, the paired difference in overall
classification correctness was not statistically significant.
This test does not establish the statistical significance
of the observed Macro F1 change.

2. NSL-KDD INDEPENDENT REPLICATION

Baseline model: RF-B-NSL
Comparison model: RF-H2-NSL
Official test observations: 22544
Target classes: 5

Validation Macro F1: {nsl_validation_f1:.6f}
Official test Macro F1: {nsl_test_f1:.6f}
Validation-to-test change: {nsl_generalisation_gap:+.6f}

Baseline-to-depth-30 Macro F1 change: {nsl_f1_change:+.6f}
Validation prediction disagreements: 0
Official test prediction disagreements: 0

The frozen depth restriction produced no observed
classification changes across either evaluated partition.
The two models need not have identical internal trees.

3. ATTACK NOVELTY AND DISTRIBUTION SHIFT

Training-seen attack-name records: {seen_count}
Training-seen correct classifications: {seen_correct}
Training-seen accuracy: {seen_correct / seen_count:.6f}

Test-only attack-name types: 17
Test-only attack-name records: {novel_count}
Test-only correct classifications: {novel_correct}
Test-only accuracy: {novel_correct / novel_count:.6f}

Test-only attack-name errors: {novel_errors}
All official-test errors: {total_errors}
Test-only share of all errors: {100 * novel_error_share:.3f}%

Official-test R2L recall: {r2l_recall:.6f}
Official-test U2R recall: {u2r_recall:.6f}

The official NSL test partition differs in attack-name
composition and family prevalence. Test-only attack names
account for a disproportionate share of classification
errors. These descriptive cohorts have different family
compositions and do not isolate a causal novelty effect.

4. COMPARISON SCOPE

CIC-IDS2017 evaluates 15 classes and an enhancement
including training-only SMOTE. NSL-KDD evaluates five
attack families and independently tests a frozen maximum-
depth configuration.

The evaluation does not demonstrate direct transfer of
a fitted CIC Random Forest to NSL-KDD. Raw Macro F1
values are not treated as equivalent across benchmarks.

5. METHODOLOGICAL LIMITATIONS

"""

for index, limitation in enumerate(
    NSL11_METHOD_LIMITATIONS,
    start=1
):
    NSL11_FINDINGS_TEXT += (
        f"{index}. {limitation}\n"
    )

# ------------------------------------------------------------
# 6. Save interpretation artefacts
# ------------------------------------------------------------

record_path = (
    NSL11_RESULTS_DIR / "NSL11_interpretation_record.json"
)

text_path = (
    NSL11_RESULTS_DIR / "NSL11_findings_summary.txt"
)

record_bytes = (
    json.dumps(
        NSL11_INTERPRETATION_RECORD,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

text_bytes = (
    NSL11_FINDINGS_TEXT.rstrip() + "\n"
).encode("utf-8")

print("\nINTERPRETATION EXPORTS")

print(
    "Interpretation JSON:",
    ns11_write_once(record_path, record_bytes)
)

print(
    "Technical findings summary:",
    ns11_write_once(text_path, text_bytes)
)

# ------------------------------------------------------------
# 7. Audit the complete evidence export
# ------------------------------------------------------------

NSL11_EVIDENCE_EXPORT_PATHS = (
    list(NSL11_EXPORTED_TABLE_PATHS)
    + list(NSL11_FIGURE_PATHS)
    + [record_path, text_path]
)

if len(NSL11_EVIDENCE_EXPORT_PATHS) != 18:
    raise RuntimeError(
        "Expected 18 analysis output artefacts."
    )

if len(set(NSL11_EVIDENCE_EXPORT_PATHS)) != 18:
    raise RuntimeError(
        "Duplicate evidence output paths."
    )

file_hashes = {}

for path in sorted(NSL11_EVIDENCE_EXPORT_PATHS):

    if not path.is_file():
        raise FileNotFoundError(path)

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"Empty evidence artefact: {path}"
        )

    file_hashes[path.name] = ns11_sha256(path)

if len(file_hashes) != 18:
    raise RuntimeError(
        "Expected 18 unique evidence fingerprints."
    )

# Independently verify stored interpretation values.

reloaded_record = ns11_load_json(record_path)

if reloaded_record != NSL11_INTERPRETATION_RECORD:
    raise RuntimeError(
        "Saved interpretation record differs."
    )

if reloaded_record["nsl_kdd"]["test_only_attack_name_errors"] != 3051:
    raise RuntimeError(
        "Novel-attack error count mismatch."
    )

if reloaded_record["nsl_kdd"]["total_test_errors"] != 5447:
    raise RuntimeError(
        "NSL official-test error total mismatch."
    )

if reloaded_record[
    "cic_ids2017"
]["macro_f1_significance_test_performed"] is not False:
    raise RuntimeError(
        "Statistical test scope mismatch."
    )

# Record the supporting evidence fingerprints.

NSL11_EXPORT_MANIFEST = {
    "notebook": "11_Cross_Benchmark_Robustness_Analysis",
    "stage": "evidence_exports",
    "table_count": 11,
    "figure_count": 5,
    "interpretation_artefact_count": 2,
    "evidence_file_count": 18,
    "evidence_file_sha256": file_hashes,
    "source_experiment_completion_files": {
        "CIC_statistics": str(
            CIC_STATS_DIR / "RF08_NOTEBOOK_COMPLETE.json"
        ),
        "NSL_preprocessing": str(
            NSL09_DIR / "NSL09_NOTEBOOK_COMPLETE.json"
        ),
        "NSL_replication": str(
            NSL10_DIR / "NSL10_NOTEBOOK_COMPLETE.json"
        )
    },
    "source_experiments_modified": False,
    "model_training_performed": False
}

manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_export_manifest.json"
)

manifest_bytes = (
    json.dumps(
        NSL11_EXPORT_MANIFEST,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

manifest_status = ns11_write_once(
    manifest_path,
    manifest_bytes
)

saved_manifest = ns11_load_json(manifest_path)

if saved_manifest != NSL11_EXPORT_MANIFEST:
    raise RuntimeError(
        "Persisted export manifest differs."
    )

for filename, expected_hash in saved_manifest[
    "evidence_file_sha256"
].items():

    path = NSL11_RESULTS_DIR / filename

    if ns11_sha256(path) != expected_hash:
        raise RuntimeError(
            f"Evidence fingerprint mismatch: {filename}"
        )

NSL11_INTERPRETATION_VERIFIED = True

print("\nINTERPRETATION SUMMARY")
print("CIC Macro F1 change:", f"{cic_f1_change:+.6f}")
print("CIC McNemar p-value:", f"{cic_mcnemar_p:.7f}")
print("NSL Macro F1 change:", f"{nsl_f1_change:+.6f}")
print(
    "NSL validation-to-test Macro F1 change:",
    f"{nsl_generalisation_gap:+.6f}"
)
print("Test-only attack records:", novel_count)
print("Test-only attack errors:", novel_errors)
print(
    "Share of NSL test errors from test-only attacks:",
    f"{100 * novel_error_share:.3f}%"
)

print("\nEVIDENCE EXPORT AUDIT")
print("CSV tables:", 11)
print("PNG figures:", 5)
print("Interpretation artefacts:", 2)
print("Fingerprinted evidence files:", len(file_hashes))
print("Export manifest:", manifest_status)
print("Evidence SHA-256 checks: PASS")
print("Interpretation record: PASS")
print("Experiment comparison scope: PASS")
print("Source experiments modified: NO")

print("\nCELL 09: PASSED")
print("Evidence package: VERIFIED")
print("Model training performed: NO")
print("=" * 72)


CELL 09 — INTERPRETATION RECORD AND EXPORT AUDIT

INTERPRETATION EXPORTS
Interpretation JSON: CREATED
Technical findings summary: CREATED

INTERPRETATION SUMMARY
CIC Macro F1 change: +0.018702
CIC McNemar p-value: 0.5384046
NSL Macro F1 change: +0.000000
NSL validation-to-test Macro F1 change: -0.453498
Test-only attack records: 3750
Test-only attack errors: 3051
Share of NSL test errors from test-only attacks: 56.012%

EVIDENCE EXPORT AUDIT
CSV tables: 11
PNG figures: 5
Interpretation artefacts: 2
Fingerprinted evidence files: 18
Export manifest: CREATED
Evidence SHA-256 checks: PASS
Interpretation record: PASS
Experiment comparison scope: PASS
Source experiments modified: NO

CELL 09: PASSED
Evidence package: VERIFIED
Model training performed: NO


In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 10 — INDEPENDENT EVIDENCE VERIFICATION
# ============================================================

from pathlib import Path
import json
import struct

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

print("=" * 72)
print("CELL 10 — INDEPENDENT EVIDENCE VERIFICATION")
print("=" * 72)

required_stages = [
    "NSL11_INPUTS_VERIFIED",
    "NSL11_METRICS_VERIFIED",
    "NSL11_CLASS_AUDIT_VERIFIED",
    "NSL11_EFFECT_AUDIT_VERIFIED",
    "NSL11_SHIFT_AUDIT_VERIFIED",
    "NSL11_SYNTHESIS_VERIFIED",
    "NSL11_TABLES_EXPORTED",
    "NSL11_FIGURES_EXPORTED",
    "NSL11_INTERPRETATION_VERIFIED"
]

for stage in required_stages:
    if not globals().get(stage, False):
        raise RuntimeError(
            f"Required analysis stage incomplete: {stage}"
        )

NSL11_INDEPENDENT_AUDIT_VERIFIED = False

# ------------------------------------------------------------
# 1. Independently verify all evidence fingerprints
# ------------------------------------------------------------

export_manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_export_manifest.json"
)

export_manifest = ns11_load_json(
    export_manifest_path
)

evidence_hashes = export_manifest[
    "evidence_file_sha256"
]

if (
    export_manifest["table_count"] != 11
    or export_manifest["figure_count"] != 5
    or export_manifest["evidence_file_count"] != 18
):
    raise RuntimeError(
        "Export manifest counts do not match."
    )

if len(evidence_hashes) != 18:
    raise RuntimeError(
        "Expected 18 fingerprinted evidence files."
    )

for filename, expected_hash in evidence_hashes.items():

    path = NSL11_RESULTS_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns11_sha256(path) != expected_hash:
        raise RuntimeError(
            f"Evidence fingerprint mismatch: {filename}"
        )

print("\nEVIDENCE FINGERPRINT VERIFICATION")
print("Evidence files:", len(evidence_hashes))
print("SHA-256 verification: PASS")

# ------------------------------------------------------------
# 2. Reload and verify every exported CSV table
# ------------------------------------------------------------

if len(NSL11_TABLE_EXPORTS) != 11:
    raise RuntimeError(
        "Expected eleven original analysis tables."
    )

for filename, expected_frame in NSL11_TABLE_EXPORTS.items():

    path = NSL11_RESULTS_DIR / filename

    archived_frame = pd.read_csv(path)

    if archived_frame.shape != expected_frame.shape:
        raise RuntimeError(
            f"Table dimensions differ: {filename}"
        )

    if list(archived_frame.columns) != list(
        expected_frame.columns
    ):
        raise RuntimeError(
            f"Table columns differ: {filename}"
        )

    pd.testing.assert_frame_equal(
        archived_frame.reset_index(drop=True),
        expected_frame.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=2e-9,
        obj=filename
    )

print("\nSAVED TABLE CONTENT VERIFICATION")
print("Tables independently reloaded:", 11)
print("Column names and row counts: PASS")
print("Numeric content reconciliation: PASS")

# ------------------------------------------------------------
# 3. Verify stored PNG image dimensions and signatures
# ------------------------------------------------------------

if len(NSL11_FIGURE_PATHS) != 5:
    raise RuntimeError("Expected five figures.")

figure_details = []

for path in NSL11_FIGURE_PATHS:

    with Path(path).open("rb") as handle:

        signature = handle.read(8)

        if signature != b"\x89PNG\r\n\x1a\n":
            raise RuntimeError(
                f"Invalid PNG signature: {path.name}"
            )

        ihdr_length = struct.unpack(
            ">I", handle.read(4)
        )[0]

        chunk_type = handle.read(4)

        if chunk_type != b"IHDR" or ihdr_length != 13:
            raise RuntimeError(
                f"Invalid PNG header: {path.name}"
            )

        width, height = struct.unpack(
            ">II", handle.read(8)
        )

    if width < 800 or height < 400:
        raise RuntimeError(
            f"Unexpected figure dimensions: {path.name}"
        )

    figure_details.append({
        "filename": path.name,
        "width": width,
        "height": height
    })

print("\nFIGURE FILE VERIFICATION")

for record in figure_details:
    print(
        f"{record['filename']}: "
        f"{record['width']} x {record['height']} PASS"
    )

# ------------------------------------------------------------
# 4. Recalculate NSL official-test predictions and metrics
# ------------------------------------------------------------

processed_test_path = (
    PROJECT_ROOT
    / "data/NSL_KDD/processed/nslkdd_test.parquet"
)

test_labels = pd.read_parquet(
    processed_test_path,
    columns=["Label"]
)["Label"].to_numpy(dtype=np.int32)

if len(test_labels) != 22544:
    raise RuntimeError(
        "Official-test target count differs."
    )

if not np.array_equal(
    np.bincount(test_labels, minlength=5),
    [9711, 7460, 2421, 2885, 67]
):
    raise RuntimeError(
        "Official-test class distribution mismatch."
    )

prediction_paths = {
    "RF-B-NSL":
        NSL10_DIR
        / "RF_B_NSL/RF_B_NSL_test_predictions.npy",
    "RF-H2-NSL":
        NSL10_DIR
        / "RF_H2_NSL/RF_H2_NSL_test_predictions.npy"
}

recomputed_results = {}
reloaded_predictions = {}

for model_name, path in prediction_paths.items():

    predictions = np.load(
        path,
        allow_pickle=False
    ).astype(np.int32)

    if predictions.shape != test_labels.shape:
        raise RuntimeError(
            f"{model_name}: prediction shape mismatch."
        )

    if np.any(predictions < 0) or np.any(predictions > 4):
        raise RuntimeError(
            f"{model_name}: invalid class prediction."
        )

    calculated = {
        "Accuracy": accuracy_score(
            test_labels, predictions
        ),
        "Macro Precision": precision_score(
            test_labels, predictions,
            labels=list(range(5)),
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            test_labels, predictions,
            labels=list(range(5)),
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            test_labels, predictions,
            labels=list(range(5)),
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            test_labels, predictions,
            labels=list(range(5)),
            average="weighted",
            zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            test_labels, predictions
        ),
        "MCC": matthews_corrcoef(
            test_labels, predictions
        )
    }

    archived_row = (
        NSL11_MODEL_METRICS.set_index("Model")
        .loc[model_name]
    )

    for metric, value in calculated.items():

        if not np.isclose(
            float(archived_row[metric]),
            value,
            rtol=0,
            atol=2e-9
        ):
            raise RuntimeError(
                f"{model_name}: {metric} mismatch."
            )

    recomputed_results[model_name] = calculated
    reloaded_predictions[model_name] = predictions

if not np.array_equal(
    reloaded_predictions["RF-B-NSL"],
    reloaded_predictions["RF-H2-NSL"]
):
    raise RuntimeError(
        "Official-test model predictions differ."
    )

baseline_predictions = reloaded_predictions[
    "RF-B-NSL"
]

correct_mask = (
    baseline_predictions == test_labels
)

correct_count = int(
    np.count_nonzero(correct_mask)
)

if correct_count != 17097:
    raise RuntimeError(
        "Official-test correct-count mismatch."
    )

confusion = confusion_matrix(
    test_labels,
    baseline_predictions,
    labels=list(range(5))
)

if confusion.shape != (5, 5):
    raise RuntimeError(
        "Unexpected test confusion matrix shape."
    )

if int(confusion.sum()) != 22544:
    raise RuntimeError(
        "Test confusion matrix total mismatch."
    )

print("\nINDEPENDENT OFFICIAL-TEST VERIFICATION")
print("Models evaluated:", len(recomputed_results))
print("Test observations:", len(test_labels))
print("Correct classifications:", correct_count)
print("Incorrect classifications:", 22544 - correct_count)
print(
    "Baseline Macro F1:",
    f"{recomputed_results['RF-B-NSL']['Macro F1']:.6f}"
)
print(
    "Tuned Macro F1:",
    f"{recomputed_results['RF-H2-NSL']['Macro F1']:.6f}"
)
print("Prediction agreement: IDENTICAL")
print("Metric reconciliation: PASS")
print("Confusion matrix: PASS")

# ------------------------------------------------------------
# 5. Independently reconcile attack-novelty cohorts
# ------------------------------------------------------------

raw_test_source = Path(
    NSL11_NSL09_MANIFEST[
        "source_files"
    ]["official_test"]["path"]
)

expected_source_hash = NSL11_NSL09_MANIFEST[
    "source_files"
]["official_test"]["sha256"]

if ns11_sha256(raw_test_source) != expected_source_hash:
    raise RuntimeError(
        "Original NSL test source fingerprint mismatch."
    )

raw_attack_names = (
    pd.read_csv(
        raw_test_source,
        header=None,
        usecols=[41],
        dtype=str
    )
    .iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.rstrip(".")
)

if len(raw_attack_names) != 22544:
    raise RuntimeError(
        "Original test source row count mismatch."
    )

test_only_names = set(
    NSL11_TEST_ONLY_ATTACKS["Attack"].astype(str)
)

test_only_mask = (
    raw_attack_names.isin(test_only_names)
    .to_numpy(dtype=bool)
)

seen_mask = ~test_only_mask

observed_counts = {
    "seen_support": int(seen_mask.sum()),
    "test_only_support": int(test_only_mask.sum()),
    "seen_correct": int(
        np.count_nonzero(correct_mask & seen_mask)
    ),
    "test_only_correct": int(
        np.count_nonzero(correct_mask & test_only_mask)
    ),
    "test_only_errors": int(
        np.count_nonzero(~correct_mask & test_only_mask)
    )
}

expected_counts = {
    "seen_support": 18794,
    "test_only_support": 3750,
    "seen_correct": 16398,
    "test_only_correct": 699,
    "test_only_errors": 3051
}

if observed_counts != expected_counts:
    raise RuntimeError(
        "Attack-novelty cohort reconciliation failed."
    )

print("\nINDEPENDENT ATTACK-NOVELTY VERIFICATION")

for name, value in observed_counts.items():
    print(f"{name}: {value}")

print("Attack-novelty cohort reconciliation: PASS")

# ------------------------------------------------------------
# 6. Verify source result files remain unmodified
# ------------------------------------------------------------

cic_source_hashes = NSL11_CIC_COMPLETION[
    "output_file_hashes"
]

nsl_source_hashes = NSL11_NSL10_MANIFEST[
    "result_file_fingerprints"
]

if len(cic_source_hashes) != 17:
    raise RuntimeError(
        "Unexpected CIC source result count."
    )

if len(nsl_source_hashes) != 22:
    raise RuntimeError(
        "Unexpected NSL source result count."
    )

for filename, expected_hash in cic_source_hashes.items():

    if ns11_sha256(
        CIC_STATS_DIR / filename
    ) != expected_hash:
        raise RuntimeError(
            f"CIC source file modified: {filename}"
        )

for relative_path, expected_hash in nsl_source_hashes.items():

    if ns11_sha256(
        PROJECT_ROOT / relative_path
    ) != expected_hash:
        raise RuntimeError(
            f"NSL source file modified: {relative_path}"
        )

NSL11_INDEPENDENT_AUDIT_VERIFIED = True

print("\nFINAL INDEPENDENT AUDIT")
print("Source CIC result files:", len(cic_source_hashes))
print("Source NSL result files:", len(nsl_source_hashes))
print("Exported evidence files:", len(evidence_hashes))
print("All exported table contents: PASS")
print("All PNG headers and dimensions: PASS")
print("NSL test metrics independently recalculated: PASS")
print("Attack-novelty counts independently recalculated: PASS")
print("Source result fingerprints: PASS")
print("Existing source experiments modified: NO")

print("\nCELL 10: PASSED")
print("Independent evidence verification: COMPLETE")
print("Model training performed: NO")
print("=" * 72)


CELL 10 — INDEPENDENT EVIDENCE VERIFICATION

EVIDENCE FINGERPRINT VERIFICATION
Evidence files: 18
SHA-256 verification: PASS

SAVED TABLE CONTENT VERIFICATION
Tables independently reloaded: 11
Column names and row counts: PASS
Numeric content reconciliation: PASS

FIGURE FILE VERIFICATION
NSL11_within_benchmark_metric_effects.png: 1980 x 1196 PASS
NSL11_nsl_validation_vs_test_class_f1.png: 2179 x 1294 PASS
NSL11_nsl_attack_family_distribution.png: 1980 x 1179 PASS
NSL11_nsl_attack_novelty_accuracy.png: 1780 x 1196 PASS
NSL11_nsl_family_novelty_recall.png: 2180 x 1196 PASS

INDEPENDENT OFFICIAL-TEST VERIFICATION
Models evaluated: 2
Test observations: 22544
Correct classifications: 17097
Incorrect classifications: 5447
Baseline Macro F1: 0.504649
Tuned Macro F1: 0.504649
Prediction agreement: IDENTICAL
Metric reconciliation: PASS
Confusion matrix: PASS

INDEPENDENT ATTACK-NOVELTY VERIFICATION
seen_support: 18794
test_only_support: 3750
seen_correct: 16398
test_only_correct: 699
test_only

In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 11 — FINAL REPRODUCIBILITY MANIFEST
# ============================================================

from pathlib import Path
import json
import hashlib

print("=" * 72)
print("CELL 11 — FINAL REPRODUCIBILITY MANIFEST")
print("=" * 72)

if not globals().get(
    "NSL11_INDEPENDENT_AUDIT_VERIFIED", False
):
    raise RuntimeError("Cell 10 must pass first.")

NSL11_FINAL_MANIFEST_VERIFIED = False

# ------------------------------------------------------------
# 1. Record verified source experiment fingerprints
# ------------------------------------------------------------

source_records = {}

for name, path in sorted(NSL11_SOURCE_FILES.items()):

    if not path.is_file():
        raise FileNotFoundError(path)

    source_records[name] = {
        "relative_path": str(
            path.relative_to(PROJECT_ROOT)
        ),
        "sha256": ns11_sha256(path)
    }

if len(source_records) != len(NSL11_SOURCE_FILES):
    raise RuntimeError(
        "Source reference count mismatch."
    )

# ------------------------------------------------------------
# 2. Verify evidence export manifest
# ------------------------------------------------------------

export_manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_export_manifest.json"
)

export_manifest = ns11_load_json(
    export_manifest_path
)

evidence_hashes = export_manifest[
    "evidence_file_sha256"
]

if len(evidence_hashes) != 18:
    raise RuntimeError(
        "Expected 18 evidence file hashes."
    )

for filename, recorded_hash in evidence_hashes.items():

    path = NSL11_RESULTS_DIR / filename

    if ns11_sha256(path) != recorded_hash:
        raise RuntimeError(
            f"Evidence file changed: {filename}"
        )

# ------------------------------------------------------------
# 3. Verify archived model references
# ------------------------------------------------------------

model_references = {}

for model_name, model_path in NSL11_MODEL_PATHS.items():

    if not (
        model_path / "saved_model.pb"
    ).is_file():
        raise RuntimeError(
            f"SavedModel unavailable: {model_name}"
        )

    model_references[model_name] = str(
        model_path.relative_to(PROJECT_ROOT)
    )

if len(model_references) != 4:
    raise RuntimeError(
        "Expected four model references."
    )

# ------------------------------------------------------------
# 4. Construct final Notebook 11 manifest
# ------------------------------------------------------------

interpretation_record_path = (
    NSL11_RESULTS_DIR
    / "NSL11_interpretation_record.json"
)

interpretation = ns11_load_json(
    interpretation_record_path
)

NSL11_FINAL_MANIFEST = {
    "notebook": "11_Cross_Benchmark_Robustness_Analysis",
    "analysis_stage": "final_evidence_verification",
    "datasets": ["CIC-IDS2017", "NSL-KDD"],

    "source_experiments": {
        "CIC-IDS2017": {
            "baseline": "RF-B-CIC",
            "comparison": "RF-E-CIC",
            "test_records": 378119,
            "classes": 15
        },
        "NSL-KDD": {
            "baseline": "RF-B-NSL",
            "comparison": "RF-H2-NSL",
            "official_test_records": 22544,
            "classes": 5
        }
    },

    "model_references": model_references,
    "source_file_fingerprints": source_records,

    "archived_cic_result_count": 17,
    "archived_nsl_result_count": 22,

    "nsl_preprocessing_fingerprint": (
        NSL11_NSL09_MANIFEST[
            "combined_artefact_fingerprint"
        ]
    ),

    "export_manifest": {
        "relative_path": str(
            export_manifest_path.relative_to(PROJECT_ROOT)
        ),
        "sha256": ns11_sha256(
            export_manifest_path
        )
    },

    "evidence_exports": {
        "csv_tables": 11,
        "png_figures": 5,
        "interpretation_files": 2,
        "fingerprinted_files": 18,
        "file_sha256": evidence_hashes
    },

    "verified_findings": {
        "cic_macro_f1_change": (
            interpretation["cic_ids2017"][
                "macro_f1_change"
            ]
        ),
        "cic_macro_recall_change": (
            interpretation["cic_ids2017"][
                "macro_recall_change"
            ]
        ),
        "cic_mcnemar_p_value": (
            interpretation["cic_ids2017"][
                "mcnemar_p_value"
            ]
        ),
        "nsl_macro_f1_change": (
            interpretation["nsl_kdd"][
                "macro_f1_change"
            ]
        ),
        "nsl_validation_to_test_macro_f1_change": (
            interpretation["nsl_kdd"][
                "validation_to_test_macro_f1_change"
            ]
        ),
        "nsl_test_prediction_disagreements": (
            interpretation["nsl_kdd"][
                "test_prediction_disagreements"
            ]
        ),
        "nsl_test_only_attack_type_count": (
            interpretation["nsl_kdd"][
                "test_only_attack_type_count"
            ]
        ),
        "nsl_test_only_attack_records": (
            interpretation["nsl_kdd"][
                "test_only_attack_records"
            ]
        ),
        "nsl_test_only_attack_errors": (
            interpretation["nsl_kdd"][
                "test_only_attack_name_errors"
            ]
        )
    },

    "methodological_controls": {
        "within_benchmark_effect_comparison": True,
        "distinct_label_spaces_acknowledged": True,
        "independent_nsl_replication": True,
        "direct_cic_to_nsl_model_transfer": False,
        "new_hyperparameter_search": False,
        "new_resampling": False,
        "new_model_training": False,
        "source_experiments_modified": False,
        "all_evidence_fingerprints_verified": True
    }
}

# ------------------------------------------------------------
# 5. Save final manifest without overwriting results
# ------------------------------------------------------------

final_manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_final_manifest.json"
)

final_manifest_content = (
    json.dumps(
        NSL11_FINAL_MANIFEST,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

manifest_status = ns11_write_once(
    final_manifest_path,
    final_manifest_content
)

# ------------------------------------------------------------
# 6. Independently verify persisted manifest
# ------------------------------------------------------------

saved_manifest = ns11_load_json(
    final_manifest_path
)

if saved_manifest != NSL11_FINAL_MANIFEST:
    raise RuntimeError(
        "Persisted final manifest differs."
    )

for name, record in saved_manifest[
    "source_file_fingerprints"
].items():

    path = PROJECT_ROOT / record["relative_path"]

    if ns11_sha256(path) != record["sha256"]:
        raise RuntimeError(
            f"Source fingerprint mismatch: {name}"
        )

for filename, recorded_hash in saved_manifest[
    "evidence_exports"
]["file_sha256"].items():

    path = NSL11_RESULTS_DIR / filename

    if ns11_sha256(path) != recorded_hash:
        raise RuntimeError(
            f"Evidence fingerprint mismatch: {filename}"
        )

if ns11_sha256(
    export_manifest_path
) != saved_manifest[
    "export_manifest"
]["sha256"]:
    raise RuntimeError(
        "Evidence export manifest fingerprint mismatch."
    )

controls = saved_manifest[
    "methodological_controls"
]

for key in [
    "direct_cic_to_nsl_model_transfer",
    "new_hyperparameter_search",
    "new_resampling",
    "new_model_training",
    "source_experiments_modified"
]:

    if controls[key] is not False:
        raise RuntimeError(
            f"Methodological control mismatch: {key}"
        )

NSL11_FINAL_MANIFEST_VERIFIED = True

print("\nFINAL MANIFEST SUMMARY")
print("Manifest:", manifest_status)
print("Datasets:", ", ".join(
    saved_manifest["datasets"]
))
print(
    "Model references:",
    len(saved_manifest["model_references"])
)
print(
    "Source reference files:",
    len(saved_manifest["source_file_fingerprints"])
)
print(
    "Evidence exports:",
    saved_manifest["evidence_exports"][
        "fingerprinted_files"
    ]
)
print(
    "Manifest SHA-256:",
    ns11_sha256(final_manifest_path)
)

print("\nREPRODUCIBILITY CHECKS")
print("Source reference fingerprints: PASS")
print("Evidence file fingerprints: PASS")
print("Export manifest fingerprint: PASS")
print("Persisted final manifest: PASS")
print("Experimental configurations: PRESERVED")
print("Source experiments modified: NO")
print("Model training performed: NO")

print("\nCELL 11: PASSED")
print("Final reproducibility manifest: VERIFIED")
print("=" * 72)


CELL 11 — FINAL REPRODUCIBILITY MANIFEST

FINAL MANIFEST SUMMARY
Manifest: CREATED
Datasets: CIC-IDS2017, NSL-KDD
Model references: 4
Source reference files: 14
Evidence exports: 18
Manifest SHA-256: 8228c75d5af35818587f58f3776dfaa6ac6d3a2ceb9a429df27ec1f3a6551e6a

REPRODUCIBILITY CHECKS
Source reference fingerprints: PASS
Evidence file fingerprints: PASS
Export manifest fingerprint: PASS
Persisted final manifest: PASS
Experimental configurations: PRESERVED
Source experiments modified: NO
Model training performed: NO

CELL 11: PASSED
Final reproducibility manifest: VERIFIED


In [ ]:

# ============================================================
# NOTEBOOK 11 — CROSS-BENCHMARK ROBUSTNESS ANALYSIS
# CELL 12 — FINAL COMPLETION RECORD
# ============================================================

from pathlib import Path
import json
import numpy as np

print("=" * 72)
print("CELL 12 — FINAL COMPLETION RECORD")
print("=" * 72)

if not globals().get(
    "NSL11_FINAL_MANIFEST_VERIFIED", False
):
    raise RuntimeError("Cell 11 must pass first.")

NSL11_NOTEBOOK_COMPLETE = False

final_manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_final_manifest.json"
)

export_manifest_path = (
    NSL11_RESULTS_DIR / "NSL11_export_manifest.json"
)

final_manifest = ns11_load_json(
    final_manifest_path
)

export_manifest = ns11_load_json(
    export_manifest_path
)

# ------------------------------------------------------------
# 1. Final numerical checks
# ------------------------------------------------------------

findings = final_manifest["verified_findings"]

if not np.isclose(
    findings["cic_macro_f1_change"],
    0.018702,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro F1 evidence mismatch."
    )

if not np.isclose(
    findings["cic_macro_recall_change"],
    0.048248,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro Recall evidence mismatch."
    )

if not np.isclose(
    findings["cic_mcnemar_p_value"],
    0.5384046,
    atol=1e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC statistical evidence mismatch."
    )

if not np.isclose(
    findings["nsl_macro_f1_change"],
    0.0,
    atol=1e-9,
    rtol=0
):
    raise RuntimeError(
        "NSL model effect mismatch."
    )

if not np.isclose(
    findings["nsl_validation_to_test_macro_f1_change"],
    -0.453498,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "NSL generalisation evidence mismatch."
    )

if findings["nsl_test_prediction_disagreements"] != 0:
    raise RuntimeError(
        "NSL prediction disagreement mismatch."
    )

if findings["nsl_test_only_attack_type_count"] != 17:
    raise RuntimeError(
        "NSL test-only attack type count mismatch."
    )

if findings["nsl_test_only_attack_records"] != 3750:
    raise RuntimeError(
        "NSL test-only attack record count mismatch."
    )

if findings["nsl_test_only_attack_errors"] != 3051:
    raise RuntimeError(
        "NSL test-only attack error count mismatch."
    )

# ------------------------------------------------------------
# 2. Final evidence file and manifest verification
# ------------------------------------------------------------

if export_manifest["evidence_file_count"] != 18:
    raise RuntimeError(
        "Evidence export count mismatch."
    )

if len(
    final_manifest["evidence_exports"]["file_sha256"]
) != 18:
    raise RuntimeError(
        "Final manifest evidence count mismatch."
    )

if (
    final_manifest["evidence_exports"]["file_sha256"]
    != export_manifest["evidence_file_sha256"]
):
    raise RuntimeError(
        "Evidence manifests do not agree."
    )

for filename, recorded_hash in export_manifest[
    "evidence_file_sha256"
].items():

    path = NSL11_RESULTS_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns11_sha256(path) != recorded_hash:
        raise RuntimeError(
            f"Final evidence verification failed: {filename}"
        )

if ns11_sha256(
    export_manifest_path
) != final_manifest["export_manifest"]["sha256"]:
    raise RuntimeError(
        "Export manifest integrity mismatch."
    )

# ------------------------------------------------------------
# 3. Create final completion record
# ------------------------------------------------------------

NSL11_COMPLETION_RECORD = {
    "notebook": "11_Cross_Benchmark_Robustness_Analysis",
    "status": "COMPLETE",

    "datasets": [
        "CIC-IDS2017",
        "NSL-KDD"
    ],

    "source_models": [
        "RF-B-CIC",
        "RF-E-CIC",
        "RF-B-NSL",
        "RF-H2-NSL"
    ],

    "completed_analysis_stages": 12,

    "evidence_exports": {
        "comparative_tables": 11,
        "figures": 5,
        "interpretation_files": 2,
        "evidence_files": 18,
        "export_manifest": 1,
        "final_manifest": 1
    },

    "primary_results": {
        "cic_macro_f1_change": (
            findings["cic_macro_f1_change"]
        ),
        "cic_macro_recall_change": (
            findings["cic_macro_recall_change"]
        ),
        "cic_mcnemar_p_value": (
            findings["cic_mcnemar_p_value"]
        ),
        "nsl_macro_f1_change": (
            findings["nsl_macro_f1_change"]
        ),
        "nsl_validation_to_test_macro_f1_change": (
            findings[
                "nsl_validation_to_test_macro_f1_change"
            ]
        ),
        "nsl_test_only_attack_types": (
            findings["nsl_test_only_attack_type_count"]
        ),
        "nsl_test_only_attack_records": (
            findings["nsl_test_only_attack_records"]
        ),
        "nsl_test_only_attack_errors": (
            findings["nsl_test_only_attack_errors"]
        )
    },

    "export_manifest_path": str(
        export_manifest_path
    ),

    "export_manifest_sha256": ns11_sha256(
        export_manifest_path
    ),

    "final_manifest_path": str(
        final_manifest_path
    ),

    "final_manifest_sha256": ns11_sha256(
        final_manifest_path
    ),

    "analysis_checks_passed": True,
    "source_experiments_modified": False,
    "model_training_performed": False,

    "next_stage": "12_Final_Research_Evidence_Consolidation"
}

completion_path = (
    NSL11_RESULTS_DIR
    / "NSL11_NOTEBOOK_COMPLETE.json"
)

completion_content = (
    json.dumps(
        NSL11_COMPLETION_RECORD,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

completion_status = ns11_write_once(
    completion_path,
    completion_content
)

# ------------------------------------------------------------
# 4. Verify persisted completion record
# ------------------------------------------------------------

saved_completion = ns11_load_json(
    completion_path
)

if saved_completion != NSL11_COMPLETION_RECORD:
    raise RuntimeError(
        "Persisted completion record mismatch."
    )

if saved_completion["status"] != "COMPLETE":
    raise RuntimeError(
        "Notebook completion status invalid."
    )

if saved_completion["analysis_checks_passed"] is not True:
    raise RuntimeError(
        "Final analysis checks not confirmed."
    )

if saved_completion["model_training_performed"] is not False:
    raise RuntimeError(
        "Unexpected model training record."
    )

if saved_completion["source_experiments_modified"] is not False:
    raise RuntimeError(
        "Unexpected source modification record."
    )

# 18 evidence files + export manifest + final manifest
# + notebook completion record.

expected_output_paths = (
    [
        NSL11_RESULTS_DIR / filename
        for filename in export_manifest[
            "evidence_file_sha256"
        ]
    ]
    + [
        export_manifest_path,
        final_manifest_path,
        completion_path
    ]
)

if len(expected_output_paths) != 21:
    raise RuntimeError(
        "Expected 21 final Notebook 11 output paths."
    )

if len(set(expected_output_paths)) != 21:
    raise RuntimeError(
        "Duplicate final output paths detected."
    )

for path in expected_output_paths:

    if not path.is_file():
        raise FileNotFoundError(path)

NSL11_NOTEBOOK_COMPLETE = True

# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\nFINAL NOTEBOOK 11 AUDIT")
print("Source experiment integrity: PASS")
print("Within-benchmark effects: PASS")
print("CIC statistical findings: PASS")
print("NSL official-test performance: PASS")
print("NSL distribution-shift analysis: PASS")
print("NSL attack-novelty analysis: PASS")
print("Methodological limitations: DOCUMENTED")
print("Evidence export fingerprints: PASS")
print("Final reproducibility manifest: PASS")
print("Completion record verification: PASS")
print("Model training performed: NO")
print("Source experiments modified: NO")

print("\nCOMPLETION RECORD")
print("Completion file:", completion_status)
print("Notebook status:", saved_completion["status"])
print(
    "Completed cells:",
    saved_completion["completed_analysis_stages"]
)
print(
    "Comparative tables:",
    saved_completion["evidence_exports"][
        "comparative_tables"
    ]
)
print(
    "Figures:",
    saved_completion["evidence_exports"]["figures"]
)
print(
    "Evidence files:",
    saved_completion["evidence_exports"][
        "evidence_files"
    ]
)
print("Final output files:", len(expected_output_paths))
print(
    "Completion record SHA-256:",
    ns11_sha256(completion_path)
)

print("\n" + "=" * 72)
print("NOTEBOOK 11: COMPLETE")
print("Final evidence package: VERIFIED")
print("Next stage: 12_Final_Research_Evidence_Consolidation")
print("=" * 72)


CELL 12 — FINAL COMPLETION RECORD

FINAL NOTEBOOK 11 AUDIT
Source experiment integrity: PASS
Within-benchmark effects: PASS
CIC statistical findings: PASS
NSL official-test performance: PASS
NSL distribution-shift analysis: PASS
NSL attack-novelty analysis: PASS
Methodological limitations: DOCUMENTED
Evidence export fingerprints: PASS
Final reproducibility manifest: PASS
Completion record verification: PASS
Model training performed: NO
Source experiments modified: NO

COMPLETION RECORD
Completion file: CREATED
Notebook status: COMPLETE
Completed cells: 12
Comparative tables: 11
Figures: 5
Evidence files: 18
Final output files: 21
Completion record SHA-256: cecd1119b30e52069d5f75c87a296b7a2bbb0a1bbd93b3dfb66e1e5ebe3e2593

NOTEBOOK 11: COMPLETE
Final evidence package: VERIFIED
Next stage: 12_Final_Research_Evidence_Consolidation
